# Tool 1: Inspect the recording parameters

This tool inspects the export parameters of the recordings of an EEG database (`.edf` files or BrainVision `.vhdr`
files). It reads the file headers only:
it does not load the signals nor check the quality of the data. It tells you whether some participants' data need to
be exported again (poor signal resolution or clipping, as explained below), and it gives an overview of the dataset
(channel labels, sampling frequencies, filters).

- **Inputs**: a study folder holding the recordings, `.edf` or BrainVision `.vhdr` files (searched in every
  subfolder), and optionally a `participants.tsv` table (columns `participant_id` and `group`).
- **Outputs**: summary tables (`.tsv`, easy to open in Excel or to reload in Python) in `reports_inspection/`,
  inside the study folder.
- **Sections**: 1. Study folder (with 1.1 Dataset general information, 1.2 Anonymization check and, when the study
  holds some, 1.3 BrainVision recordings), then one
  section per channel type ticked (EEG and EOG by default): 2. EEG, 3. EOG, 4. ECG, 5. EMG.

<details style="border:1px solid #bbb; border-radius:4px; background:#f7f7f7; padding:8px 12px; margin:10px 0;"><summary style="cursor:pointer; font-weight:bold;">Details <span style="font-size:85%; font-weight:normal; color:#666;">(click to open)</span></summary>

**The EDF format.** EDF (European Data Format) is a standard format for storing multichannel biological and physical
signals: https://www.edfplus.info/. It was first published in 1992, and an upgraded version (EDF+) was released in
2003 (adding discontinuous recordings, annotations, stimuli and events in a UTF-8 format). EDF is not compressed:
each sample is stored as a 16-bit integer (2¹⁶ possible values), scaled between the physical minimum and maximum
that you set at export. These two bounds are the dynamic range of the channel.

**Why inspect the export.** At export in the EDF format (with software like Profusion from Compumedics), many
parameters need to be set per recorded channel, such as the channel label, sampling frequency, filters, units and
dynamic range. Exporting is tedious and time-consuming, so mistakes are easily made. To avoid them, some software
can store a montage that applies your pre-defined parameters directly (see the tutorial
`tutorials/2_create_EEG_montage_Profusion.docx` for Compumedics Profusion). Inspecting these parameters is also
needed when you work on an existing dataset, to make sure that every participant's data can be used, especially
when the data come from several sleep clinics.

**How it reads the files.** This tool reads the information from the `.edf` headers itself, instead of loading the
data with existing packages: packages such as pyEDFlib or MNE are either too rigid (unable to read some files) or do
not return all the information (bounds of the dynamic range, filter settings, etc.).

**BrainVision recordings.** A BrainVision recording is three files that name one another (`.vhdr` header, `.vmrk`
markers, `.eeg` signal), so one of them cannot be renamed alone. The tool lists the `.vhdr`, and Section 1.3 shows what
its text files give: online reference, recording pauses, gain of the auxiliary inputs, electrode impedances. The format
declares no physical range and has no patient field, so the dynamic range, polarity and anonymization checks are
marked *not applicable*. The resolution check is too when the samples are stored as floating-point numbers.

Developed and tested on PSG databases exported from Compumedics Profusion, with groups as subfolders or nights as
file-name suffixes.

</details>

In [ ]:
#%% Import packages and define custom functions
# Import cell
try:
    import os
    import re
    import chardet
    import warnings
    import traceback
    import numpy as np
    import pandas as pd
    from pathlib import Path
    import ipywidgets as widgets
    from ipyfilechooser import FileChooser
    from IPython.display import display, HTML, clear_output
except ImportError as e:
    print('❌ Import error, a package is missing:', e)

# Shared EDF reading (tools/edf_io_lib.py, also used by tool 0), probed from the repo root (the canonical launch
# folder) or from tools/, so the import works whatever folder Voila is launched from.
import sys
for _p in [os.getcwd(), os.path.join(os.getcwd(), 'tools'),
           os.path.join(os.path.dirname(os.getcwd()), 'tools'), os.path.join(os.getcwd(), '..', 'tools')]:
    if os.path.isfile(os.path.join(_p, 'edf_io_lib.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
from edf_io_lib import extract_filter_value, check_anonymization
import recording_lib as RL   # one way to read a recording, whatever its format
import study_files_lib as SF   # which files of the study are recordings
import ui_lib as UI   # the look shared by the Voila pages

# custom function to get the sampling frequency out of a dataframe (the df needs to have 'subject' and 'channel' as columns)
def get_sf(df, subject, channel):
    df_sf = df[(df['subject'] == subject) & (df['channel'] == channel)]
    if not df_sf.empty:
        return df_sf.iloc[0]['sampling_frequency']
    else:
        return None

# function to create a widget slider to select the configuration to inspect
def mk_config_slider(value = 1, min = 1, max = 5):
    config_slider = widgets.IntSlider(
    value=value,
    min=min,
    max=max,
    step=1,
    description='Selected configuration:',
    style={'description_width': '150px'},   # increase description width (to adjust based on the description)
    layout=widgets.Layout(width='400px'),   # to adjust widget size
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
    )
    return config_slider

# function to print the configuration of a dataset parameter
def print_config(i, config_dict, param):
    # get the key and value from the dictionary
    idx = i - 1
    # get participant ID
    value = list(config_dict.values())  
    v = value[idx]  
    # get configuration
    key = list(config_dict.keys())
    k = key[idx]
    
    # print info
    print(f'Selected configuration: # {i}')
    print(f'\t{len(k)} {param}: {k}')
    print(f'\t{len(v)} participants: {v}')

# function to create a scrollable box for long output (e.g., cell loading the data) 
def print_in_scrollable_box(text, height=300, font_size="12px"):
    display(HTML(f'<pre style="overflow-y:scroll; height:{height}px; border:1px solid black; padding:10px; font-size:{font_size};">{text}</pre>'))

# Only a format storing integers between two declared bounds (EDF) has a dynamic range and a polarity to check, and only
# integer samples have a fixed step (a resolution) to check.
NO_BOUNDS = 'the format declares no physical bounds'
FLOAT_SAMPLES = 'samples stored as floating-point numbers have no fixed step'

def has_physical_bounds(df):
    return pd.to_numeric(df['physical_min'], errors='coerce').notna()

def not_applicable(df, checked, what, why):
    """Says which channels a check cannot look at. True when that is all of them: nothing else is to be said."""
    n_out = int((~checked).sum())
    if n_out:
        print(f'\n{what}: not applicable to {n_out} / {len(df)} channel(s) ({why}).')
    return n_out > 0 and n_out == len(df)


#%% select study folder (the rest of the code will be encapsulated in the folder selection so that voila waits for the selection to run the rest of cells)
# folder selector
chooser = FileChooser(os.getcwd())
chooser.title = UI.picker_title('Study folder')
chooser.show_only_dirs = True

# channel types to inspect (checked = inspected). Robust detection = transducer type OR channel-name list.
cb_eeg = widgets.Checkbox(value=True,  description='EEG', indent=False, style={'description_width': 'initial'})
cb_eog = widgets.Checkbox(value=True,  description='EOG', indent=False, style={'description_width': 'initial'})
cb_ecg = widgets.Checkbox(value=False, description='ECG', indent=False, style={'description_width': 'initial'})
cb_emg = widgets.Checkbox(value=False, description='EMG', indent=False, style={'description_width': 'initial'})
types_box = widgets.VBox([
    widgets.HTML('<b>Channel types to inspect:</b> <i>(EEG &amp; EOG by default; tick ECG/EMG to add them)</i>'),
    widgets.HBox([cb_eeg, cb_eog, cb_ecg, cb_emg])
])

# result display area
out = widgets.Output()

# skip + run controls
skip_existing = widgets.Checkbox(
    value=True,
    description='Skip files already inspected',
    style={'description_width': 'initial'}
)
existing_files_info = widgets.HTML(value='')
run_button = UI.button('Run inspection', 'run')
progress, progress_row = UI.progress_bar('Files:')

section1 = widgets.HTML(UI.section_html(1, 'Study folder'))
display(section1)

# update the info line: how many of the folder's EDF files are already in FULL_summary_table_edf.tsv
def _update_existing_info(*_):
    if not getattr(chooser, 'selected_path', None):
        existing_files_info.value = ''
        return
    try:
        folder = Path(chooser.selected_path)
        edf_files = SF.list_recordings(folder)
        n_total = len(edf_files)
        if n_total == 0:
            existing_files_info.value = UI.msg('warn', 'No recordings (.edf, .vhdr) found in selected folder.')
            return
        full_path = folder / 'reports_inspection' / 'FULL_summary_table_edf.tsv'
        n_done = 0
        if full_path.exists():
            df_prev = pd.read_csv(str(full_path), sep='\t', dtype={'subject': str, 'sub_num': str, 'group': str, 'session': str})
            df_prev = df_prev.loc[:, ~df_prev.columns.str.startswith('Unnamed')]
            if 'path' in df_prev.columns:
                existing_paths = set(os.path.normcase(p) for p in df_prev['path'].astype(str))
                n_done = sum(os.path.normcase(str(f)) in existing_paths for f in edf_files)
        n_new = (n_total - n_done) if skip_existing.value else n_total
        existing_files_info.value = (
            f'{n_done} / {n_total} recording(s) already inspected. '
            f'{n_new} recording(s) will be read on run.'
        )
        _conflicts = SF.recording_conflicts_text(folder)
        if _conflicts:
            existing_files_info.value += '<br>' + UI.msg('warn', _conflicts)
    except Exception as e:
        existing_files_info.value = UI.msg('error', f'Error checking existing inspection: {e}')

# main function to run the inspection when a folder have been selected
def run_inspection(_):
    out.clear_output()
    with out:
        chooser.folder_path = chooser.selected_path
        print("📁 Selected Path:", chooser.folder_path)
        
        # get the edf file list 
        chooser.edf_files = SF.list_recordings(chooser.folder_path)
        if not chooser.edf_files:
            print(f"⚠️ There is no recording (.edf, .vhdr) in your folder")
        else:
            print(f"\nThere is {len(chooser.edf_files)} recording(s) in your folder!")
            _conflicts = SF.recording_conflicts_text(chooser.folder_path)
            if _conflicts:
                print(f"⚠️ {_conflicts}")
        
        # check the existence and/or create the summary folder that will receive the summary tables and the report
        chooser.summary_path = f'{chooser.folder_path}/reports_inspection'
        if not os.path.exists(chooser.summary_path):
            os.makedirs(chooser.summary_path)
            readme_text = (
                "# reports_inspection - Output files\n\n"
                "This folder is created automatically by the EDF inspection tools (Dream-Toolkit).\n"
                "It contains summary tables (.tsv) and HTML reports generated from the EDF files in the parent study folder.\n\n"
                "## Output files\n\n"
                "| File | Tool | Description |\n"
                "|------|------|-------------|\n"
                "| FULL_summary_table_edf.tsv | inspect_edf, perdataset | One row per EDF file - all header fields |\n"
                "| EDF_inspection_report.html | perdataset | Human-readable HTML inspection report |\n"
                "| EDF_perParticipant_report.html | perparticipant | Per-participant HTML report |\n"
                "| EEG_summary_table.tsv | all | EEG channel overview |\n"
                "| EEG_missing_edf.tsv | all | EDF files where EEG channels could not be detected |\n"
                "| EEG_suspect_edf.tsv | all | EDF files with an unexpected number of EEG channels |\n"
                "| EEG_inverted_polarity_edf.tsv | all | EEG channels with suspected inverted polarity |\n"
                "| EEG_bad_dynamic_range_edf.tsv | all | EEG channels with a dynamic range below threshold |\n"
                "| EEG_bad_resolution_edf.tsv | all | EEG channels with a resolution above threshold |\n"
                "| EOG_*.tsv | all | Same checks for EOG channels |\n"
                "| ECG_*.tsv | all | Same checks for ECG channels (only if ECG is selected) |\n"
                "| EMG_*.tsv | all | Same checks for EMG channels (only if EMG is selected) |\n"
                "| anonymization_check_edf.tsv | all | Per-file header anonymization status; flags non-anonymized patient names |\n"
                "| recording_notes.tsv | Voila | BrainVision recordings: online reference, pauses, what the [Comment] could not give |\n"
                "| impedance_flags.tsv | Voila | BrainVision recordings: electrodes above the impedance threshold or without a measure |\n"
                "| failed_edf_read.tsv | all | Recordings that could not be read |\n\n"
                "## Tools that generate this folder\n\n"
                "- **1_inspect_edf_voila.ipynb** - Interactive UI (run with `voila tools/1_inspect_edf_voila.ipynb`). Recommended for non-programmers.\n"
                "- **1_inspect_edf_perdataset.py** - Script version; processes a full dataset folder in one shot.\n"
                "- **1_inspect_edf_perparticipant.py** - Script version; generates one report per participant.\n"
            )
            with open(f'{chooser.summary_path}/FILES_DESCRIPTION.md', 'w', encoding='utf-8') as rf:
                rf.write(readme_text)
            print("\nCreated reports_inspection folder at: " + chooser.summary_path)
        else:
            print("\nSummary folder already exists. \nSummary tables will be updated (merged) at: \n" + chooser.summary_path)

        # get variables from the chooser widget
        folder_path = chooser.folder_path
        summary_path = chooser.summary_path
        edf_files = chooser.edf_files

        # --- skip / merge setup: which files are already inspected? ---
        full_table_path = f'{summary_path}/FULL_summary_table_edf.tsv'
        existing_full = pd.DataFrame()
        if os.path.exists(full_table_path):
            try:
                # reload with the SAME str dtypes the fresh-read path produces, so merged rows are type-identical
                # (a TSV round-trip otherwise turns e.g. sampling_frequency '256' into int 256, breaking sorted(set(...)))
                _str_cols = ['subject', 'group', 'session', 'path', 'sub_folder', 'sub_num', 'pre_fn_comp', 'post_fn_comp', 'channel', 'transducer_type', 'dimension', 'sampling_frequency', 'physical_min', 'physical_max', 'patient_name']
                existing_full = pd.read_csv(full_table_path, sep='\t', dtype={c: str for c in _str_cols})
                existing_full = existing_full.loc[:, ~existing_full.columns.str.startswith('Unnamed')]
            except Exception as _e:
                print(f"⚠️ Could not read existing FULL_summary_table_edf.tsv ({_e}); it will be rebuilt from scratch.")
                existing_full = pd.DataFrame()
        existing_paths = set(os.path.normcase(p) for p in existing_full['path'].astype(str)) if 'path' in existing_full.columns else set()

        # decide which files to read this run
        if skip_existing.value:
            to_read = [f for f in edf_files if os.path.normcase(str(f)) not in existing_paths]
        else:
            to_read = list(edf_files)
        processed_paths = {os.path.normcase(str(f)) for f in to_read}
        n_skipped = len(edf_files) - len(to_read)
        print(f"\nSkip already inspected: {'ON' if skip_existing.value else 'OFF'}")
        print(f"Files to read this run: {len(to_read)} / {len(edf_files)}" + (f" ({n_skipped} skipped)" if n_skipped else ""))
        if not to_read and existing_full.empty:
            print("\n⚠️ Nothing to inspect (no new files and no existing table). Stopping.")
            return

        # check if there is a participants.tsv file to get different groups or sessions
        # if there is not a participants.tsv we will try to infer groups from subfolder organization or filename components (additional part from subject number)
        # e.g. subfolders can define groups within the study folder,
        # and file-name suffixes can define nights ("session")
        table_found = False
        for root, dirs, files in os.walk(folder_path):
            if 'participants.tsv' in files:
                table_found = True        
                print(f"Table containing participants information found at: {os.path.join(root, 'participants.tsv')} ")
                subj_table_path = os.path.join(root, 'participants.tsv')
        
        found_group = False
        if table_found:
            subj_table = pd.read_csv(subj_table_path, sep = '\t', dtype={'participant_id': str, 'group': str})
            if "group" in subj_table.columns:
                found_group = True
                print(f"We will extract participant's group from it")
            else:
                print(f"No column 'group' was found in the table")
                print(f"Group will be inferred from subfolder organization or subfolder component") 
            
        else:
            print(f"No table containing participants information (labelled 'participants.tsv') was found")
            print(f"If you have a table, please rename it 'participants.tsv' (and make sure you have columns labelled 'participant_id' and 'group' if any)")
            print(f"In the meantime, we will infer participant's group from subfolder organization or filename component")
            subj_table = pd.DataFrame()

        # initialyse list of dataframe to store file info, that will be concatenated at the end (this is better for performance)
        df_list = []
        # Initialize an empty list for files that could not be read
        failed_list = []
        anon_rows = []  # anonymization check results, one dict per file
        bv_rows = []    # what a BrainVision header says beyond its channels, one dict per recording (Section 1.3)
        imp_rows = []   # their electrode impedances, one dict per electrode
        # initialize output for a dynamic display (with a scroll box)
        output = ""
        dynamic_out = widgets.Output()
        display(dynamic_out)

        # Loop over the files to read (new files only when skip is on)
        progress.max = max(1, len(to_read))
        progress.value = 0
        progress.bar_style = 'info'
        for e, edf_path in enumerate(to_read):
            with dynamic_out:
                output += (f'file {e+1}/{len(to_read)}, currently opening file: {edf_path}\n')
                dynamic_out.clear_output(wait=True)
                print_in_scrollable_box(output, font_size = "12px")
                
                # read file with the custom function
                try:
                    edf_header = RL.read_header(edf_path)
                    is_brainvision = RL.file_format(edf_path) == 'brainvision' 
                    # BrainVision has no patient field to check (Section 1.2)
                    anon_info = None if is_brainvision else check_anonymization(edf_header['patient_id'], edf_path.stem)
                    
                    # get subject name (corresponding to file_name)
                    sub_name = edf_path.stem
                    
                    # get subject group (from the parent folder, for databases with one subfolder per patient group)
                    sub_folder = edf_path.parent.name # get the parent folder of the subject file (path)
                    
                    # create df from signal info
                    df = RL.channel_table(edf_header)
                    # An online reference that is not in the file is named in Section 1.3, not inspected as a channel
                    if is_brainvision:
                        df = df[df['in_file']].copy()
                        
                    # theoretical resolution (edf are 16bit files so the eeg signal can take 2^16 values within the dynamic range)
                    df['res_theoretical'] = (abs(pd.to_numeric(df['physical_min']))+abs(pd.to_numeric(df['physical_max'])))/pow(2,16)
                    # BrainVision declares its step directly, which only means one for samples stored as integers
                    if is_brainvision:
                        df['res_theoretical'] = np.nan if 'FLOAT' in edf_header['binary_format'] else pd.to_numeric(df['resolution'], errors='coerce')
                    # turn theoretical resolution to uV if dimension is mV (if no dimension, it is a mess)
                    df.loc[df['dimension'].str.contains('mv', case=False, na=False), 'res_theoretical'] *= 1000
                    
                    # get filtering info in different columns
                    df['lowpass']   = df['prefiltering'].apply(lambda x: extract_filter_value(x, 'LP'))
                    df['highpass']  = df['prefiltering'].apply(lambda x: extract_filter_value(x, 'HP'))
                    df['notch']  = df['prefiltering'].apply(lambda x: extract_filter_value(x, 'NOTCH'))
                    
                    # add subject info in the dataframe
                    df['subject'] = sub_name
                    df['sub_folder'] = sub_folder
                    df['group'] = np.nan # initialyze column 'group' with NaN
                    # get group from participants table if any (else group will be inferred from subfolder or filename extension later)
                    if found_group:
                        df['group'] = subj_table.loc[subj_table['participant_id'] == sub_name, 'group'].iloc[0]
        
                    # extract filename component before and after subject number (so we assume subject name contains at least incrementing numbers that are at the beginning of the file name)  
                    #   ^       → start of string  
                    # (.*?)     → group 1: as few chars as possible, up to the first digit  
                    # (\d+)     → group 2: the number itself  
                    # (.*)      → group 3: the rest of the string  
                    # $         → end of string
                    pre_comp = sub_num = post_comp = np.nan
                    pattern = re.compile(r'^(.*?)(\d+)(.*)$')
                    m = pattern.match(sub_name)
                    if m:
                        pre_comp = m.group(1) or np.nan
                        sub_num = m.group(2) or np.nan
                        post_comp = m.group(3) or np.nan
                    df['pre_fn_comp'] = pre_comp
                    df['post_fn_comp'] = post_comp
                    df['sub_num'] = sub_num
                    
                    df['patient_name'] = anon_info['name_subfield'] if anon_info else ''
                    df['path'] = str(edf_path)
                    df['session'] = np.nan # session will be inferred later from file name component
                    
                    # what only a BrainVision header gives per channel (no such column in an EDF study)
                    bv_columns = []
                    if is_brainvision:
                        df['impedance_kohm'] = df['channel'].map(edf_header['impedance_kohm'])
                        df['gradient'] = df['gradient'].map(lambda g: f'{g[0]:g} {g[1]}' if g and g[0] is not None else '')
                        bv_columns = ['resolution', 'impedance_kohm', 'gradient']

                    # select only the columns of interest
                    df = df[['subject', 'group', 'session', 'path', 'sub_folder', 'sub_num', 'pre_fn_comp', 'post_fn_comp', 'channel', 'transducer_type', 'dimension', 'sampling_frequency', 
                         'highpass', 'lowpass', 'notch', 'physical_min', 'physical_max', 'res_theoretical', 'patient_name'] + bv_columns]
                    
                    if anon_info:
                        anon_rows.append({
                            'subject'           : sub_name,
                            'path'              : str(edf_path),
                            'patient_id'        : anon_info['patient_id'],
                            'name_subfield'     : anon_info['name_subfield'],
                            'patient_id_format' : anon_info['patient_id_format'],
                            'header_anonymized' : anon_info['header_anonymized'],
                            'name_in_filename'  : anon_info['name_in_filename'],
                            'anon_warning'      : anon_info['anon_warning'],
                        })
                    if is_brainvision:
                        bv_rows.append({
                            'subject'           : sub_name,
                            'path'              : str(edf_path),
                            'online_reference'  : edf_header['online_reference'],
                            'reference_in_file' : all(edf_header['in_file']),
                            'n_segments'        : edf_header['n_segments'],
                            'start_known'       : bool(edf_header['start_date']),
                            'binary_format'     : edf_header['binary_format'],
                            'comment_warning'   : edf_header['comment_warning'],
                        })
                        imp_rows += [{'subject': sub_name, 'path': str(edf_path), 'electrode': electrode, 'impedance_kohm': value,
                                      'reading': edf_header['impedance_text'].get(electrode, ''), 'measured_at': edf_header['impedance_time']}
                                     for electrode, value in edf_header['impedance_kohm'].items()]
                    # store subject data
                    df_list.append(df)
            
                except UnicodeDecodeError:
                    err = f"⚠️ Encoding problem for {edf_path}\n"
                    output += err
                    clear_output(wait=True)
                    print_in_scrollable_box(output, font_size="12px")
                    failed_list.append((edf_path, 'encoding'))
                except Exception as read_error:
                    # tb = traceback.format_exc()
                    err = f"❌ Unexpected problem for {edf_path} : {read_error}\n"
                    output += err
                    clear_output(wait=True)
                    print_in_scrollable_box(output, font_size="12px")
                    failed_list.append((edf_path, 'other'))
            progress.value = e + 1
        UI.finish_progress(progress, n_failed=len(failed_list))
           
        # concatenate dataframe into one and only
        with warnings.catch_warnings(): # this is to skip a warning not affecting our operation
            warnings.simplefilter("ignore", FutureWarning)
            new_rows = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()
        # keep existing rows for files NOT re-read this run (merge/replace on the 'path' key)
        if not existing_full.empty and 'path' in existing_full.columns:
            existing_keep = existing_full[~existing_full['path'].astype(str).map(os.path.normcase).isin(processed_paths)]
        else:
            existing_keep = pd.DataFrame()
        _parts = [d for d in (existing_keep, new_rows) if not d.empty]
        if not _parts:
            print("\n⚠️ No data to summarise. Stopping.")
            return
        with warnings.catch_warnings():
            # the only all-NA columns here are session/group (recomputed just below) and the
            # filter columns (only written to the TSV), so the deprecated all-NA dtype behavior is harmless
            warnings.simplefilter("ignore", FutureWarning)
            df_full = pd.concat(_parts, ignore_index=True)
        # re-derive dataset-level inferred columns so newly added files are classified consistently
        df_full['session'] = np.nan
        if not found_group:
            df_full['group'] = np.nan
        
        # save the failed list if not empty:
        failed_df = pd.DataFrame(failed_list, columns=['path', 'reason'])
        failed_out = f'{summary_path}/failed_edf_read.tsv'
        try:
            failed_df['path'] = failed_df['path'].astype(str)
            if os.path.exists(failed_out):
                prev_failed = pd.read_csv(failed_out, sep='\t', dtype=str)
                prev_failed = prev_failed.loc[:, ~prev_failed.columns.str.startswith('Unnamed')]
                if 'path' in prev_failed.columns:
                    prev_failed = prev_failed[~prev_failed['path'].astype(str).map(os.path.normcase).isin(processed_paths)]
                failed_df = pd.concat([prev_failed, failed_df], ignore_index=True)
            if not failed_df.empty:
                failed_df.to_csv(failed_out, sep='\t', index=False)
            elif os.path.exists(failed_out):
                os.remove(failed_out)
        except Exception as _e:
            print(f"⚠️ Could not update failed_edf_read.tsv: {_e}")
        if not failed_df.empty:
            print(f'\nSaving the list of files that could not be read to: \n{summary_path}/failed_edf_read.tsv')    
        #____________________________________________________________________________________________
        
        #%% 1.2 General info of the dataset___________________________________________________________
        section12 = widgets.HTML(UI.subsection_html('1.1', 'Dataset general information (# participants, groups, recorded sensors)'))
        display(section12)
        
        # get group information, from participants.tsv file, sub_folder, or filename component
        print("Get group information:")
        if df_full['group'].isna().all():
            print("The column 'group' is empty, we will infer group from subfolder, if any...")
            if len(df_full['sub_folder'].unique()) > 1:
                df_full['group'] = df_full['sub_folder']
                print(">>> Group inferred from folders within the database <<<")
            else:
                print("There is no distinct folders for groups.")
                print("Trying to infer group from filename component...")
                # looping across subject number (and not subject filename) to test if there are multiple filename components per subject (to disentangle groups from session)  
                count_precomp = np.zeros(len(df_full['sub_num'].unique()))
                count_postcomp = np.zeros(len(df_full['sub_num'].unique()))
                for sn, sub_num in enumerate(df_full['sub_num'].unique()):
                    df_sub = df_full[df_full['sub_num'] == sub_num]
                    count_precomp[sn] = len(df_sub['pre_fn_comp'].unique())
                    count_postcomp[sn] = len(df_sub['post_fn_comp'].unique())
                # fn component is a group if within subject there is only one component, but there are multiple components between subject
                # 1st, try for component before the subject number, 2nd try for component after the subject number 
                if len(df_full['pre_fn_comp'].unique()) > 1 and count_precomp.mean() == 1:
                    df_full['group'] = df_full['pre_fn_comp']
                    print(">>> Group inferred from filename component (before subject number) <<<")
                elif len(df_full['post_fn_comp'].unique()) > 1 and count_postcomp.mean() == 1:
                    df_full['group'] = df_full['post_fn_comp']
                    print(">>> Group inferred from filename component (after subject number) <<<")
                else:
                    print("Did not succeed to identify group from filename component.")
                    print("It seems that there is only one group in the study!")
        else:
            print(">>> Group information coming from participants.tsv <<<")
        
        print("\nGet session information")
        if len(df_full['pre_fn_comp'].unique()) > 1 and count_precomp.mean() > 1:
            print(">>> Session inferred from filename component (before subject number) <<<")
            df_full['session'] = df_full['pre_fn_comp']
        elif len(df_full['post_fn_comp'].unique()) > 1 and count_postcomp.mean() > 1:
            print(">>> Session inferred from filename component (after subject number) <<<")
            df_full['session'] = df_full['post_fn_comp']
        else:
            print("It seems that there is only one session in the study")
        
        # save summary table containing full info
        df_full.to_csv(f'{summary_path}/FULL_summary_table_edf.tsv', sep = '\t', index=False)
        print(f'\nSaving full informations from the dataset to:\n{summary_path}/FULL_summary_table_edf.tsv')

        # end-of-run summary
        n_failed_run = len(failed_list)
        print("\n=== Inspection run complete ===")
        print(f"- Files read this run : {len(to_read) - n_failed_run}" + (f" ({n_failed_run} failed)" if n_failed_run else ""))
        print(f"- Files skipped       : {n_skipped}")
        print(f"- Total files in table: {df_full['path'].nunique() if 'path' in df_full.columns else len(df_full['subject'].unique())}")
        
        # ---- from here, the DISPLAYED analysis concerns only the files read this run ----
        # saved summary tables stay the full cumulative dataset via _merge_save (rows for the
        # files processed this run are replaced; rows for other files are kept).
        processed_subjects = set(df_full[df_full['path'].astype(str).map(os.path.normcase).isin(processed_paths)]['subject'].astype(str))
        df_full = df_full[df_full['path'].astype(str).map(os.path.normcase).isin(processed_paths)].copy()
        if df_full.empty:
            print("\nNo new files were read: nothing new to display. Existing summary tables are unchanged.")
            return

        def _merge_save(df_new, out_path):
            # write df_new, merging with any existing file so the saved table stays the full
            # cumulative dataset (rows for processed_subjects replaced, other rows kept)
            try:
                if os.path.exists(out_path):
                    prev = pd.read_csv(out_path, sep='\t', dtype=str)
                    prev = prev.loc[:, ~prev.columns.str.startswith('Unnamed')]
                    if 'subject' in prev.columns:
                        prev = prev[~prev['subject'].astype(str).isin(processed_subjects)]
                    with warnings.catch_warnings():
                        # df_out is only written to the TSV, so the deprecated all-NA dtype behavior is harmless
                        warnings.simplefilter("ignore", FutureWarning)
                        df_out = pd.concat([prev, df_new], ignore_index=True)
                else:
                    df_out = df_new
                df_out.to_csv(out_path, sep='\t', index=False)
            except Exception as _e:
                print(f"⚠️ Could not merge-save {os.path.basename(out_path)}: {_e}")
                df_new.to_csv(out_path, sep='\t', index=False)

        print("\n\nDataset information:")
        print(f"- Number of files: {len(df_full['subject'].unique())}")
        print(f"- Number of participants: {len(df_full['sub_num'].unique())}")
        print(f"- Number of groups: {len(df_full['group'].unique())}")
        print(f"- Number of sessions: {len(df_full['session'].unique())}")
        
        if len(df_full['group'].unique()) > 1:
            print("\nParticipants per groups:")
            print(df_full.drop_duplicates().groupby('group').agg(n_subjects=('subject', 'nunique')))
    
        print('\nFull recorded sensors configuration of your database (across participants): ')
        ch_output = '\n'.join(df_full['channel'].unique())
        print_in_scrollable_box(ch_output, height = 150)
        #____________________________________________________________________________________________

        #%% 1.3 Anonymization check______________________________________________________________
        section13 = widgets.HTML(UI.subsection_html('1.2', 'Anonymization check', '''
        <p>Checks whether the EDF header field <em>Local Patient ID</em> has been anonymized.<br>
        The EDF+ format encodes the patient name as the 4th sub-field
        (Compumedics example: <code>LASTNAME_FIRSTNAME</code>, anonymized: <code>X_X</code>).<br>
        A file is flagged if any non-placeholder characters remain after stripping
        <code>X</code>, <code>x</code>, <code>_</code> and spaces from the name sub-field.
        Additionally, if a real name is detected, each name token (≥3 chars) is searched
        in the file name (case-insensitive) to detect PII leaking into the file name.</p>
        '''))
        display(section13)
        try:
            anon_df = pd.DataFrame(anon_rows) if anon_rows else pd.DataFrame(
                columns=['subject','path','patient_id','name_subfield','patient_id_format','header_anonymized','name_in_filename','anon_warning'])
            n_anon   = int(anon_df['header_anonymized'].sum()) if not anon_df.empty else 0
            n_total_anon = len(anon_df)
            warned   = anon_df[anon_df['anon_warning'] != ''] if not anon_df.empty else pd.DataFrame()
            if bv_rows:
                display(HTML(UI.msg('info', f'<b>Not applicable to {len(bv_rows)} BrainVision recording(s):</b> the format has no '
                                            'patient field. Their files can still point to a session: the .vmrk holds the recording '
                                            'date and the [Comment] of the .vhdr the serial number of the amplifier. Think of them '
                                            'before sharing the data.')))
            if warned.empty:
                if n_total_anon or not bv_rows:
                    display(widgets.HTML(
                        UI.msg('ok', f'<b>All {n_anon}/{n_total_anon} inspected file(s) have an anonymized header.</b>')
                    ))
            else:
                display(widgets.HTML(
                    UI.msg('warn', f'<b>{len(warned)}/{n_total_anon} file(s) have a non-anonymized or suspect header:</b>')
                ))
                warn_lines = '\n'.join(
                    f"  [{row['anon_warning']}]  {row['subject']}  |  patient_id: {row['patient_id']}"
                    for _, row in warned.iterrows()
                )
                print_in_scrollable_box(warn_lines, height=200)
            if n_total_anon or not bv_rows:
                _merge_save(anon_df, f'{summary_path}/anonymization_check_edf.tsv')
                print(f'\nAnonymization check saved to:\n{summary_path}/anonymization_check_edf.tsv')
        except Exception as _anon_e:
            print(f'⚠️ Could not display anonymization check: {_anon_e}')
        #____________________________________________________________________________________________

        #%% 1.3 BrainVision recordings (only when the study holds some)_____________________________
        if bv_rows:
            section14 = widgets.HTML(UI.subsection_html('1.3', 'BrainVision recordings', """
            <p>What the text files of these recordings say beyond the channel tables. It comes from the free-text [Comment] of
            the .vhdr and from the .vmrk: what a recording does not state is simply missing.</p>
            """))
            display(section14)
            try:
                bv_df = pd.DataFrame(bv_rows)
                _merge_save(bv_df, f'{summary_path}/recording_notes.tsv')

                # The online reference: all the channels are recorded against it, and it is usually not stored
                for (_ref, _in_file), _part in bv_df.fillna({'online_reference': ''}).groupby(['online_reference', 'reference_in_file']):
                    if not _ref:
                        display(HTML(UI.msg('warn', f'The [Comment] names no online reference ({len(_part)} recording(s)): it cannot be recreated.')))
                    elif _in_file:
                        display(HTML(UI.msg('info', f'<b>Online reference: {_ref}</b>, in the file ({len(_part)} recording(s)).')))
                    else:
                        display(HTML(UI.msg('info', f'<b>Online reference: {_ref}</b>, not in the file ({len(_part)} recording(s)). '
                                                    'Tool 2 lists it so that it can be kept: it is recreated when the data are re-referenced.')))

                # A pause: the samples no longer follow the clock
                _paused = bv_df[bv_df['n_segments'] > 1]
                if not _paused.empty:
                    display(HTML(UI.msg('warn', f'<b>{len(_paused)} recording(s) were paused</b> (several New Segment markers in the .vmrk): '
                                                'their samples no longer follow the clock, so scored events exported with clock times '
                                                'cannot be placed on them: ' + ', '.join(_paused['subject']))))
                _undated = bv_df[~bv_df['start_known']]
                if not _undated.empty:
                    display(HTML(UI.msg('warn', f'{len(_undated)} recording(s) give no start date in the .vmrk: ' + ', '.join(_undated['subject']))))
                for _, _row in bv_df[bv_df['comment_warning'] != ''].iterrows():
                    display(HTML(UI.msg('warn', f"{_row['subject']}: {_row['comment_warning']}")))

                # The header knows the amplifier, not what sits before its auxiliary input
                _aux = df_full[df_full['gradient'].eq('1 mV/mV') & df_full['transducer_type'].isin(['ECG', 'EMG'])]
                if not _aux.empty:
                    display(HTML(UI.msg('info', f"<b>{', '.join(sorted(_aux['channel'].unique()))}</b> sit on auxiliary inputs, with a gain of "
                                                '1 mV/mV in the header. An adapter that amplifies them (a BIP2AUX has a gain of 100) is not '
                                                'known to the header: their amplitudes would then be too large by that factor. Tools 3bis '
                                                'and 7 check the amplitude from the signal.')))

                # Impedances, the last measure of each file. A text instead of a number ('Out of Range!', 'not connected') = no measure.
                imp_df = pd.DataFrame(imp_rows, columns=['subject', 'path', 'electrode', 'impedance_kohm', 'reading', 'measured_at'])
                imp_df['impedance_kohm'] = pd.to_numeric(imp_df['impedance_kohm'], errors='coerce')
                _no_imp = sorted(set(bv_df['subject']) - set(imp_df['subject']))
                if _no_imp:
                    display(HTML(UI.msg('info', f'No impedances in the [Comment] of {len(_no_imp)} recording(s): ' + ', '.join(_no_imp))))
                if not imp_df.empty:
                    display(widgets.HTML(UI.markdown_html("""
                    <h4>Impedances</h4>
                    <p>Electrodes above the threshold or without a measure (the file says e.g. <i>Out of Range!</i>, <i>not connected</i>)
                    are listed and saved to a summary table. A good impedance does not promise a good channel: an electrode can come
                    off during the recording.</p>
                    """)))
                    imp_thres = widgets.BoundedFloatText(
                        value=20,
                        min=0,
                        max=10000,
                        step=1,
                        style={'description_width': '180px'},
                        layout=widgets.Layout(width='260px'),
                        description='Impedance threshold (kΩ):',
                        disabled=False
                    );

                    def check_impedance(threshold):
                        _no_measure = imp_df['impedance_kohm'].isna()
                        flagged = imp_df[_no_measure | (imp_df['impedance_kohm'] > threshold)].copy()
                        flagged['status'] = np.where(flagged['impedance_kohm'].isna(), flagged['reading'], f'above {threshold:g} kOhm')
                        if not flagged.empty:
                            print(f'\n>>> {len(flagged)} electrode(s) above {threshold:g} kΩ or without a measure, from {len(imp_df)} in {imp_df["subject"].nunique()} recording(s) <<<')
                            print(flagged[['subject', 'electrode', 'impedance_kohm', 'status']].to_string(index=False))
                        else:
                            print(f'\n>>> No electrode above {threshold:g} kΩ <<<')
                        _merge_save(flagged, f'{summary_path}/impedance_flags.tsv')
                        print(f'\nSaving the flagged electrodes to:\n{summary_path}/impedance_flags.tsv \n(will be empty if none)')

                    widgets.interact(check_impedance, threshold=imp_thres);
            except Exception as _bv_e:
                print(f'⚠️ Could not display the BrainVision notes: {_bv_e}')
        #____________________________________________________________________________________________

        # channel types selected via the checkboxes shown above the Run button
        include_eeg = cb_eeg.value
        include_eog = cb_eog.value
        include_ecg = cb_ecg.value
        include_emg = cb_emg.value
        if not (include_eeg or include_eog or include_ecg or include_emg):
            print('\n⚠️ No channel type selected, only the general dataset and anonymization checks were run.')
        if include_eeg:
            #%% 2. Select only the EEGs__________________________________________________________________
            section2 = widgets.HTML(UI.section_html(2, 'Inspect EEG'))
            display(section2)
            
            # define common EEG label from the 10-10 (MCN) convention plus older 10-20 labels (T3/T4/T5/T6 = modern T7/T8/P7/P8)
            COMMON_EEG_label = r'\bFp1\b|\bFpz\b|\bFp2\b|\bAF7\b|\bAF3\b|\bAFz\b|\bAF4\b|\bAF8\b|\bF7\b|\bF5\b|\bF3\b|\bF1\b|\bFz\b|\bF2\b|\bF4\b|\bF6\b|\bF8\b|\bFT7\b|\bFC5\b|\bFC3\b|\bFC1\b|\bFCz\b|\bFC2\b|\bFC4\b|\bFC6\b|\bFT8\b|\bT7\b|\bC5\b|\bC3\b|\bC1\b|\bCz\b|\bC2\b|\bC4\b|\bC6\b|\bT8\b|\bTP7\b|\bCP5\b|\bCP3\b|\bCP1\b|\bCPz\b|\bCP2\b|\bCP4\b|\bCP6\b|\bTP8\b|\bP7\b|\bP5\b|\bP3\b|\bP1\b|\bPz\b|\bP2\b|\bP4\b|\bP6\b|\bP8\b|\bPO7\b|\bPO5\b|\bPO3\b|\bPOz\b|\bPO4\b|\bPO6\b|\bPO8\b|\bO1\b|\bOz\b|\bO2\b|\bM1\b|\bM2\b|\bT3\b|\bT4\b|\bT5\b|\bT6\b|\bEEG\b|\bA2\b|\bA1\b'
            # select only EEG channels and return a warning if the number of participant is smaller/higher
            mask_ch = df_full['transducer_type'].str.contains(r'\bEEG\b|\bAGAGCL ELECTRODE\b', case = False, na=False) | df_full['channel'].str.contains(COMMON_EEG_label, case = False, na=False) # create a mask that returns true for lines containing either EEG/AGAGCL ELECTRODE in the transducer_type column or containing a common EEG label in the channel column
            df_ch = df_full[mask_ch]
            # remove the emg channels that were captured with the AGAGCL ELECTRODE transducer type 
            df_ch = df_ch[~df_ch['channel'].str.contains(r'emg|ecg|eog|ekg|chin|menton', case=False, na=False)] # the ~ allows to not select the selection (like ! in matlab)
            
            # Check if the number of participants with only EEG is the same as df_full. 
            # If not, it might be because the transducer type was no correctly detected. 
            # One possibility is to add the type of transducer to the condition line 2 of this cell.
            if len(df_full['subject'].unique()) > len(df_ch['subject'].unique()):
                # identify missing subjects
                missing_sub = set(df_full['subject'].unique()) - set(df_ch['subject'].unique())
                print('\n!!! There is less participants in the dataset with only EEGs !!!')
                print(f'Missing participants: {missing_sub}')
                print("\nEither these participants don't have EEGs.")
                print("Or the transducer type was not correctly detected.")
                # get df of missing sub to save and inspect
                df_miss = df_full[df_full['subject'].isin(missing_sub)]
                _merge_save(df_miss, f'{summary_path}/EEG_missing_edf.tsv')
                print(f'\nSaving informations from missing participants to:\n{summary_path}/EEG_missing_edf.tsv')
                print('Please inspect the file, and specifically the column transducer_type')
            elif len(df_full['subject'].unique()) < len(df_ch['subject'].unique()):
                print('\n!!! There is more participants in the dataset with only EEGs !!!')
                print('This should not be the case.')
                print('Please inspect what is happening in a code editor (spyder..), or ask Yvan.')
                more_sub = set(df_ch['subject'].unique()) - set(df_full['subject'].unique())
                df_more = df_ch[df_ch['subject'].isin(more_sub)]
                _merge_save(df_more, f'{summary_path}/EEG_suspect_edf.tsv')
                print(f'\nSaving informations from suspect participants to:\n{summary_path}/EEG_suspect_edf.tsv')
            
            # saving info from eeg
            _merge_save(df_ch, f'{summary_path}/EEG_summary_table.tsv')
            print(f'\nSaving informations from EEGs to:\n{summary_path}/EEG_summary_table.tsv')
            #____________________________________________________________________________________________

            #%% 2.1 Inspect EEG configurations___________________________________________________________
            section21 = widgets.HTML(UI.subsection_html('2.1', 'Inspect EEG configurations', """
            <p>By EEG configurations, here, we mean the number of EEG channels and their labeling.
            <br>In practice, most EEG studies rely on a single configuration, since all data are typically recorded at the same location, with the same system, and by the same experimenter.
            <br>In multicentric datasets, there will likely be many EEG configurations (specific to each recording center).</p>
            <p>Knowing your channels' configurations will allow you to select the subset of channels for your analyses (and later re-harmonize the channel labeling if needed).</p>
            <p>In classic polysomnographic EEG, there should be at least 4 EEG (F3, C3, O1 and A2 used as the reference) (or less frequently F4, C4, O2 and A1).</p>
            <p>Depending on your planned analyses, if a given configuration lacks those channels, you will either need to re-export the data (provided those channels were originally recorded) or exclude the participant.</p>
            <p>Below, we check how many EEG configurations your dataset contains.</p>
            """))
            display(section21)
            
            # get the EEG configuration per participant 
            ch_per_sub = df_ch.groupby('subject')['channel'].apply(lambda x: tuple(sorted(set(x))))
            
            # identify the channel configuration of each participant and store them in a dict to print per channel config
            ch_config_dict = {}
            for config in ch_per_sub.unique():
                sub = ch_per_sub[ch_per_sub == config].index.tolist()
                ch_config_dict[config] = sub
            
            if len(ch_config_dict) > 1:
                print('\n>>> There is multiple EEG configurations in your dataset! <<<')    
                print(f'\n\tNumber of different configuration: {len(ch_config_dict)}\n')
            else:
                print('\n>>> There is only one EEG configuration in your dataset! <<<\n')

            if len(ch_config_dict)>=1:
                # widget to select the configuration of interest
                config_ch_slider = mk_config_slider(value = 1, min = 1, max = len(ch_config_dict))
                
                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=ch_config_dict, param="channels"), i=config_ch_slider);
            else: 
                print("No EEG configuration found")
            #____________________________________________________________________________________________

            #%% 2.2 Inspect sampling frequency___________________________________________________________
            section22 = widgets.HTML(UI.subsection_html('2.2', 'Inspect EEG sampling frequency', """
            <p>The sampling frequency is the number of recorded samples per time unit (expressed in Hz). It is set at the acquisition.
            <br>Ideally, you expect to have only one sampling frequency for all the EEGs and participants.
            <br>In multicentric dataset, you might end up with different sampling frequencies across participants (specific to each recording center).</p> 
            <p>If you have multiple sampling frequencies across participants in your dataset, we recommend that you harmonize your dataset by downsampling your data to the lowest sampling frequency before your analyses.</p>
            <p>Below, we check how many different sampling frequencies your dataset contains.</p>
            <p><em>Side note: With multiple sampling frequencies within participants (that can happen for EEG and EOG), each EEG analysis software behaves differently. For example:
            <br>&#x2022; MNE python will automatically upsample channels to the highest sampling frequency (with .edf/.bdf/.gdf format)
            <br>&#x2022; Fieldtrip will load only a subset of channels (with the sampling frequency the most represented)
            </em></p>
            """))
            display(section22)
            
            # the sampling frequency configuration
            sf_per_sub = df_ch.groupby('subject')['sampling_frequency'].apply(lambda x: tuple(sorted(set(x))))
            # identify the sampling frequency configuration of each participant and store them in a dict to print per sampling configuration config
            sf_config_dict = {}
            for config in sf_per_sub.unique():
                sub = sf_per_sub[sf_per_sub == config].index.tolist()
                sf_config_dict[config] = sub
            
            # print info per sf configuration (maybe print it only for multiple config)
            if len(sf_config_dict) > 1:
                print('\n>>> There is multiple sampling frequency for EEGs in your dataset! <<<')    
                print(f'\n\tNumber of different sampling frequency configuration: {len(sf_config_dict)}\n')
                print('Quick overlook of the EEGs associated to sampling frequencies:')
                for s, sf in enumerate(df_ch['sampling_frequency'].unique()):
                    # select only rows with the current sf
                    df_sf = df_ch[df_ch['sampling_frequency'] == sf].copy()
                    print(f'\n{sf} Hz: {df_sf["channel"].unique()}\n')
            else:
                print(f'\n>>> There is only one sampling frequency for EEGs in your dataset: {df_ch["sampling_frequency"].unique()} <<<\n')

            if len(sf_config_dict)>=1:
                # widget to select the configuration of interest
                config_sf_slider = mk_config_slider(value = 1, min = 1, max = len(sf_config_dict))
                
                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=sf_config_dict, param="sampling frequencies"), i=config_sf_slider);
            else:
                print("No EEG sampling frequency found")
            #____________________________________________________________________________________________

            #%% 2.3 Inspect EEG filters__________________________________________________________________
            section23 = widgets.HTML(UI.subsection_html('2.3', 'Inspect EEG filters', """
            <p>When we visualize EEG data, signals are classically filtered by the software (like compumedics).
            <br>For analyses, we classically apply high-pass (to remove very low frequency), low-pass (to remove high frequency), and notch (to remove electric noise) filters.
            <br>When we export the data, we can specify whether we want the data to be filtered or not.
            <br>A good practice is to export the data without any filter, so that you can apply filters later according to your analyses.
            <br>However, for whole-night recordings, we recommend to export the data with a high-pass filter of 0.01 Hz in order to remove slow drift on such long recordings.</p>
            <p>If you have multiple filter configurations, we recommend re-exporting the data without filters if possible.</p>
            <p>Below, we check which filters were applied and counts how many different ones were used when exporting your dataset.</p>
            """))
            display(section23)
            
            if len(df_ch['highpass'].unique())+len(df_ch['lowpass'].unique())+len(df_ch['notch'].unique()) == 3:
                print('\n>>> All EEGs have the same filters! <<<')
            elif len(df_ch['highpass'].unique())+len(df_ch['lowpass'].unique())+len(df_ch['notch'].unique()) > 3:
                print('\n>>> Filters are not fully consistent in EEGs across the dataset! <<<')
            else:
                print('\n>>> There may have been a problem in reading the filters. Here is the output: <<<')
            
            # Get the list of participants with different filtering parameters
            # 1st replace NaN because groupby does not like NaN
            df_filt = df_ch.copy()
            df_filt[['lowpass', 'highpass', 'notch']] = df_filt[['lowpass', 'highpass', 'notch']].fillna('missing')
            
            config_filters = (
                df_filt.groupby(['lowpass', 'highpass', 'notch'])['subject']
                .apply(lambda x: sorted(set(x)))
                .reset_index(name = 'subjects')
            )
            
            # print filter configuration
            print(f'\n\tNumber of different EEG filters configurations: {len(config_filters)}\n')

            if len(config_filters)>=1:
                # widget to select the configuration of interest
                config_filter_slider = mk_config_slider(value = 1, min = 1, max = len(config_filters))
                
                # function to rpint filters configurations
                def print_filters(config_slider):
                    # get the info from the dataframe
                    idx = config_slider - 1
                    sID = config_filters.iloc[idx]['subjects']
                    hpass = config_filters.iloc[idx]['highpass']
                    lpass = config_filters.iloc[idx]['lowpass']
                    notch = config_filters.iloc[idx]['notch']
                    
                    # print info
                    print(f'Selected configuration: # {config_slider}')
                    print(f'\tFilters configuration: highpass: {hpass}; lowpass: {lpass}; notch: {notch}')
                    print(f'\t{len(sID)} participants: {sID}')
                
                widgets.interact(print_filters, config_slider = config_filter_slider);
            else:
                print("No EEG filters found")
            #____________________________________________________________________________________________

            #%% 2.4 Inspect units in the dataset_________________________________________________________
            section24 = widgets.HTML(UI.subsection_html('2.4', 'Inspect EEG units', """
            <p>At the exportation, channels can be imported in different units.
            <br>Each analysis software will handle units differently, so it can be helpful to know which units your dataset contains.
            </p> 
            <p>&#x2022; MNE python will automatically detect the units and convert the data to Volt. However, if the unit is not read correctly, the data will <b>not</b> be converted (e.g. "UV" is not interpredted as µV, therefore data are not converted to Volt)
            <br>&#x2022; Fieldtrip is loading the data with their unit of origin, so you might want to convert all channels to the same unit before your analysis
            </p>
            <p>Below, we check how many different units your dataset contains.</p>
            """))
            display(section24)

            if len(df_ch['dimension'].unique()) == 1:
                print(f'\n>>> All EEGs have the same unit: {df_ch["dimension"].unique()} <<<\n')
            elif len(df_ch['dimension'].unique()) > 1:
                print('\n>>> Multiple units were found! <<<')
                print(f'\n\tNumber of different units configurations: {len(df_ch["dimension"].unique())}\n')
                print('Quick overlook of EEGs associated to units:')
                for u, unit in enumerate(df_ch['dimension'].unique()):
                    # select only rows with the current sf
                    df_unit = df_ch[df_ch['dimension'] == unit].copy()
                    print(f'\n{unit}: {df_unit["channel"].unique()}')
                print(f'\n')
                
            # print the different configuration of units 
            # if info about sf configuration is needed
            unit_per_sub = df_ch.groupby('subject')['dimension'].apply(lambda x: tuple(sorted(set(x))))
            ch_per_unit = df_ch.groupby('dimension')['channel'].apply(lambda x: tuple(sorted(set(x))))
            # identify the sampling frequency configuration of each participant and store them in a dict to print per sampling configuration config
            unit_config_dict = {}
            for config in unit_per_sub.unique():
                sub = unit_per_sub[unit_per_sub == config].index.tolist()
                unit_config_dict[config] = sub

            if len(unit_config_dict)>=1:
                # widget to select the configuration of interest
                config_unit_slider = mk_config_slider(value = 1, min = 1, max = len(unit_config_dict))
                
                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=unit_config_dict, param="Units"), i=config_unit_slider);
            else:
                print("No EEG unit found")
            #____________________________________________________________________________________________

            #%% 2.5 Inspect EEG signal inversion_________________________________________________________
            section25 = widgets.HTML(UI.subsection_html('2.5', 'Inspect EEG signal inversion', """
            <p>Some softwares (e.g. profusion from compumedics) allows to invert the polarity of the exported data. It can be extremely confusing and can lead to wrong results.
            <br>Here, we inspect if the signal is inverted by checking if the minimum physical boundary is higher than the maximum physical boundary.
            <br>For .edf file, the physical boundaries are values that are set when exporting the data by specifying the scale of the data.
            <br>In profusion (from compumedics) a scale of 1mV will lead to a min physical boundary of -500 µV and a max physical boundary of +500 µV.
            </p>
            <p>For other EEG format and software, the dynamical range might be set before recording (e.g. to be specified in the montage) and can't be changed at the exportation.
            </p>
            <p> Below, we check, for each EEG channel, if the minimum physical boundary is greater than the maximum physical boundary, and saves a table containing the channels with inverted polarity.</p>
            """))
            display(section25)
            
            # select rows where the physical min is greater than the physical max
            df_inv = df_ch[df_ch['physical_min'] > df_ch['physical_max']]
            
            if not df_inv.empty:
                print('\n>>> Inverted polarity detected in EEGs! <<<')
                print(f'{df_inv.shape[0]} EEGs have an inverted polarity (from {df_ch.shape[0]} EEGs in {len(edf_files)} recordings)')
                print(df_inv[['subject', 'channel', 'dimension', 'physical_min', 'physical_max']])
            elif not not_applicable(df_ch, has_physical_bounds(df_ch), 'Inverted polarity', NO_BOUNDS):
                print('\n>>> No inverted polarity was detected in EEGs <<<')
            _merge_save(df_inv, f'{summary_path}/EEG_inverted_polarity_edf.tsv')
            print(f'\nSaving informations from inverted polarity EEGs to:\n{summary_path}/EEG_inverted_polarity_edf.tsv \n(will be empty if no inverted polarity)')
            #____________________________________________________________________________________________

            #%% 2.6 Inspect EEG dynamic range and resolution_____________________________________________
            section26 = widgets.HTML(UI.subsection_html('2.6', 'Inspect EEG dynamic range and resolution', """
            <p>The EDF format stores signals in 16 bits, meaning that each sample can take 65&nbsp;536 discrete values (2<sup>16</sup>).
            <br>In order to convert these values into real EEG amplitudes, a dynamic range (a minimum and maximum value) must be defined when exporting the data.
            <br>Each sample is then given a value between this minimum and maximum (among the 2<sup>16</sup> levels).
            </p>
            <p>This choice of dynamic range can lead to two opposing problems:</p>
                <ul>
                    <li>
                        <strong>Clipping:</strong><br>
                        If the dynamic range is too small, certain signal amplitudes exceed the limits.<br>
                        The exceeding values are then cut off (therefore “locked” at the min/max), and information is lost.<br>
                        Example of data with a dynamic range of ± 100 µV:<br>
                        <img src="images/clipped.png" width="250"/>
                    </li>
                    <li>
                        <strong>Loss of resolution:</strong><br>
                        If the dynamic range is too large, the 65&nbsp;536 levels are spread over a too-wide amplitude.<br>
                        Each quantization step then becomes too large, and small variations in signal amplitude are no longer visible with precision.<br>
                        Example of data with a resolution of 30 µV:<br>
                        <img src="images/low_resolution.png" width="250">
                    </li>
                    Example of clean data (dynamic range = ± 500 µV; resolution = 0.01 µV:<br>
                    <img src="images/clean.png" width="250"/>
                </ul>
            """))
            display(section26)
            #---------------------------
            section261 = widgets.HTML(UI.markdown_html("""
            <h4>Dynamic range</h4>
            <p>Typical physiological EEG data (good quality) varies from ± 500 µV.
            <br>Below, we check if the dynamic range physical boundaries are lower than 500 µV (± 250 µV).
            <br>You can change the dynamic range threshold with the widget.
            <br>Detected bad channels are saved to a summary table.</p>
            """))
            display(section261)
            
            dr_thres = widgets.BoundedFloatText(
                value=500,
                min=0,
                max=5000,
                step=0.1,
                style={'description_width': '200px'},  # augmente la largeur de la description
                layout=widgets.Layout(width='270px'),   # ajuste la taille totale du widget si besoin
                description='Dynamic range threshold (µV):',
                disabled=False
            );
            
            
            def check_bad_dr(threshold):
                dr_mask = (df_ch['res_theoretical']*pow(2,16) <= threshold) & has_physical_bounds(df_ch)
                bad_dr = df_ch[dr_mask]
                
                if not bad_dr.empty:
                    print(f'\n>>> Dynamic range <= {threshold} µV detected in EEGs! <<<\n')
                    print(f'{bad_dr.shape[0]} EEGs detected (from {df_ch.shape[0]} EEGs in {len(edf_files)} recordings)')
                    print(bad_dr[['subject', 'channel', 'dimension', 'physical_min', 'physical_max', 'res_theoretical']])
                elif not not_applicable(df_ch, has_physical_bounds(df_ch), 'Dynamic range', NO_BOUNDS):
                    print(f'\n>>> No EEG with a dynamic range <= {threshold} µV was detected! <<<')
                _merge_save(bad_dr, f'{summary_path}/EEG_bad_dynamic_range_edf.tsv')
                print(f'\nSaving informations from bad dynamic range EEGs to:\n{summary_path}/EEG_bad_dynamic_range_edf.tsv \n(will be empty if no bad resolution)')
            
            widgets.interact(check_bad_dr, threshold = dr_thres);
            #---------------------------
            #---------------------------
            section262 = widgets.HTML(UI.markdown_html("""
            <h4>Resolution</h4>
            <p>The theoretical resolution of .edf file is the minimum amplitude variation that can be recorded between two samples (influenced by the dynamic range, as stated above).
            <br>Below, we detect EEG channels that have a resolution higher than 0.1 µV.
            <br>You can change the resolution threshold with the widget.
            <br>Channels with a lower resolution than the threshold are saved to a summary table.</p>
            """))
            display(section262)

            # res_theo have been converted to uV, but if dimension was not read or not indicated in the headers, it might not work. I might need to add something more robust
            r_thres = widgets.BoundedFloatText(
                value=0.1,
                min=0,
                max=10.0,
                step=0.1,
                style={'description_width': '150px'},  # augmente la largeur de la description
                layout=widgets.Layout(width='230px'),   # ajuste la taille totale du widget si besoin
                description='Resolution threshold (µV):',
                disabled=False
            );
            
            # define a function to interact with the widget
            def check_bad_res(threshold):
                r_mask = df_ch['res_theoretical'] >= threshold
                bad_res = df_ch[r_mask]
                
                if not bad_res.empty:
                    print(f'\n>>> EEGs with a resolution >= {threshold} µV detected! <<<')
                    print(f'{bad_res.shape[0]} EEGs detected (from {df_ch.shape[0]} EEGs in {len(edf_files)} recordings)')
                    print(bad_res[['subject', 'channel', 'dimension', 'physical_min', 'physical_max', 'res_theoretical']])
                elif not not_applicable(df_ch, df_ch['res_theoretical'].notna(), 'Resolution', FLOAT_SAMPLES):
                    print(f'\n>>> No EEG with a resolution >= {threshold} µV was detected! <<<')
                _merge_save(bad_res, f'{summary_path}/EEG_bad_resolution_edf.tsv')
                print(f'\nSaving informations from bad resolution EEGs to:\n{summary_path}/EEG_bad_resolution_edf.tsv \n(will be empty if no bad resolution)')
            
            widgets.interact(check_bad_res, threshold=r_thres);
            #---------------------------
            #____________________________________________________________________________________________

        if include_eog:
            #%% 3. Select only the EOGs__________________________________________________________________
            section3 = widgets.HTML(UI.section_html(3, 'Inspect EOG', """
            <p>This section follows the same structure, logic, and outputs as section 2. Inspect EEG.
            <br>Hence, the code cells are not commented (please refer to section 2. if you need a refresher).
            </p>
            """))
            display(section3)

            # select only EOGs and return a warning if the number of participant is smaller/higher
            mask_eog = df_full['transducer_type'].str.contains(r'\bEOG', case = False, na=False) | df_full['channel'].str.contains(r'EOG|LOC|ROC|\bE1\b|\bE2\b', case = False, na=False) # create a mask that returns true for lines containing either EOG in the channel column
            df_eog = df_full[mask_eog]
            # remove the emg channels that were captured with the AGAGCL ELECTRODE transducer type 
            # df_eog = df_eog[~df_eog['channel'].str.contains(r'emg|ecg|eeg|a1|a2', case=False, na=False)] # the ~ allows to not select the selection (like ! in matlab)
            
            # Check if the number of participants with only EOG is the same as df_full. 
            # If not, it might be because the transducer type was no correctly detected. 
            # One possibility is to add the type of transducer to the condition line 2 of this cell.
            if len(df_full['subject'].unique()) > len(df_eog['subject'].unique()):
                # identify missing subjects
                missing_sub = set(df_full['subject'].unique()) - set(df_eog['subject'].unique())
                print('\n!!! There is less participants in the dataset with only EOGs !!!')
                print(f'Missing participants: {missing_sub}')
                print("\nEither these participants don't have EOGs.")
                print("Or the transducer type was not correctly detected.")
                # get df of missing sub to save and inspect
                df_eogmiss = df_full[df_full['subject'].isin(missing_sub)]
                _merge_save(df_eogmiss, f'{summary_path}/EOG_missing_edf.tsv')
                print(f'\nSaving informations from missing participants to:\n{summary_path}/EOG_missing_edf.tsv')
                print('Please inspect the file, and specifically the column transducer_type')
            elif len(df_full['subject'].unique()) < len(df_eog['subject'].unique()):
                print('\n!!! There is more participants in the dataset with only EOGs !!!')
                print('This should not be the case.')
                print('Please inspect what is happening in a code editor (spyder..), or ask Yvan.')
                more_sub = set(df_eog['subject'].unique()) - set(df_full['subject'].unique())
                df_more = df_eog[df_eog['subject'].isin(more_sub)]
                _merge_save(df_more, f'{summary_path}/EOG_suspect_edf.tsv')
                print(f'\nSaving informations from suspect participants to:\n{summary_path}/EOG_suspect_edf.tsv')
            
            # saving info from EOG
            _merge_save(df_eog, f'{summary_path}/EOG_summary_table.tsv')
            print(f'\nSaving informations from EOGs to:\n{summary_path}/EOG_summary_table.tsv')
            #____________________________________________________________________________________________

            #%% 3.1 Inspect EOG configurations___________________________________________________________
            section31 = widgets.HTML(UI.subsection_html('3.1', 'Inspect EOG configurations'))
            display(section31)

            # get the EOG configuration per participant 
            eog_per_sub = df_eog.groupby('subject')['channel'].apply(lambda x: tuple(sorted(set(x))))
            
            # identify the EOG configuration of each participant and store them in a dict to print per EOG config
            eog_config_dict = {}
            for config in eog_per_sub.unique():
                sub = eog_per_sub[eog_per_sub == config].index.tolist()
                eog_config_dict[config] = sub
            
            if len(eog_config_dict) > 1:
                print('\n>>> There is multiple EOG configurations in your dataset! <<<')    
                print(f'\n\tNumber of different configuration: {len(eog_config_dict)}\n')
            else:
                print('\n>>> There is only one EOG configuration in your dataset! <<<')

            if len(eog_config_dict)>=1:
                # widget to select the configuration of interest
                config_eog_slider = mk_config_slider(value = 1, min = 1, max = len(eog_config_dict))
                
                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=eog_config_dict, param="Channels"), i=config_eog_slider);
            else:
                print("No EOG configuration found")
            #____________________________________________________________________________________________
            
            #%% 3.2 Inspect EOG sampling frequencies_____________________________________________________
            section32 = widgets.HTML(UI.subsection_html('3.2', 'Inspect EOG sampling frequencies'))
            display(section32)

            # the sampling frequency configuration
            sfeog_per_sub = df_eog.groupby('subject')['sampling_frequency'].apply(lambda x: tuple(sorted(set(x))))
            # identify the sampling frequency configuration of each participant and store them in a dict to print per sampling configuration config
            sfeog_config_dict = {}
            for config in sfeog_per_sub.unique():
                sub = sfeog_per_sub[sfeog_per_sub == config].index.tolist()
                sfeog_config_dict[config] = sub
            
            # print info per sf configuration (maybe print it only for multiple config)
            if len(sfeog_config_dict) > 1:
                print('\n>>> There is multiple sampling frequency for EOGs in your dataset! <<<')    
                print(f'\n\tNumber of different sampling frequency configuration: {len(sfeog_config_dict)}\n')
                print('Quick overlook of the EOGs associated to sampling frequencies:')
                for s, sf in enumerate(df_eog['sampling_frequency'].unique()):
                    # select only rows with the current sf
                    df_sf = df_eog[df_eog['sampling_frequency'] == sf].copy()
                    print(f'\n{sf} Hz: {df_sf["channel"].unique()}')
            else:
                print(f'\n>>> There is only one sampling frequency for EOGs in your dataset: {df_eog["sampling_frequency"].unique()} <<<')

            if len(sfeog_config_dict)>=1:
                # widget to select the configuration of interest
                config_sfeog_slider = mk_config_slider(value = 1, min = 1, max = len(sfeog_config_dict))
                
                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=sfeog_config_dict, param="Sampling frequencies"), i=config_sfeog_slider);
            else:
                print("No EOG sampling frequency found")
            #____________________________________________________________________________________________
            
            #%% 3.3 Inspect EOG filters__________________________________________________________________
            section33 = widgets.HTML(UI.subsection_html('3.3', 'Inspect EOG filters'))
            display(section33)

            if len(df_eog['highpass'].unique())+len(df_eog['lowpass'].unique())+len(df_eog['notch'].unique()) == 3:
                print('\n>>> All EOGs have the same filters! <<<')
            elif len(df_eog['highpass'].unique())+len(df_eog['lowpass'].unique())+len(df_eog['notch'].unique()) > 3:
                print('\n>>> Filters are not fully consistent across the dataset! <<<')
            else:
                print('\n>>> There may have been a problem in reading the filters. Here is the output: <<<')
            
            # Get the list of participants with different filtering parameters
            # 1st replace NaN because groupby does not like NaN
            df_eogfilt = df_eog.copy()
            df_eogfilt[['lowpass', 'highpass', 'notch']] = df_eogfilt[['lowpass', 'highpass', 'notch']].fillna('missing')
            
            config_eogfilters = (
                df_eogfilt.groupby(['lowpass', 'highpass', 'notch'])['subject']
                .apply(lambda x: sorted(set(x)))
                .reset_index(name = 'subjects')
            )

            # print filter configuration
            print(f'\n\tNumber of different EOG filters configurations: {len(config_eogfilters)}\n')

            if len(config_eogfilters)>=1:
                # widget to select the configuration of interest
                config_eogfilter_slider = mk_config_slider(value = 1, min = 1, max = len(config_eogfilters))
                
                # function to rpint filters configurations
                def print_eogfilters(config_slider):
                    # get the info from the dataframe
                    idx = config_slider - 1
                    sID = config_eogfilters.iloc[idx]['subjects']
                    hpass = config_eogfilters.iloc[idx]['highpass']
                    lpass = config_eogfilters.iloc[idx]['lowpass']
                    notch = config_eogfilters.iloc[idx]['notch']
                    
                    # print info
                    print(f'Selected configuration: # {config_slider}')
                    print(f'\tFilters configuration: highpass: {hpass}; lowpass: {lpass}; notch: {notch}')
                    print(f'\t{len(sID)} participants: {sID}')
                
                widgets.interact(print_eogfilters, config_slider = config_eogfilter_slider);
            else:
                print("No EOG filters found")
            #____________________________________________________________________________________________
            
            #%% 3.4 Inspect EOG units____________________________________________________________________
            section34 = widgets.HTML(UI.subsection_html('3.4', 'Inspect EOG units'))
            display(section34)

            if len(df_eog['dimension'].unique()) == 1:
                print(f'\n>>> All EOGs have the same unit: {df_eog["dimension"].unique()} <<<\n')
            elif len(df_eog['dimension'].unique()) > 1:
                print('\n>>> Multiple units were found! <<<')
                print(f'\n\tNumber of different EOG units configurations: {len(df_eog["dimension"].unique())}\n')
                print('Quick overlook of EOGs associated to units:')
                for u, unit in enumerate(df_eog['dimension'].unique()):
                    # select only rows with the current sf
                    df_unit = df_eog[df_eog['dimension'] == unit].copy()
                    print(f'\n{unit}: {df_unit["channel"].unique()}')
                
            
            # print the different configuration of units 
            # if info about sf configuration is needed
            eogunit_per_sub = df_eog.groupby('subject')['dimension'].apply(lambda x: tuple(sorted(set(x))))
            eog_per_unit = df_eog.groupby('dimension')['channel'].apply(lambda x: tuple(sorted(set(x))))
            # identify the sampling frequency configuration of each participant and store them in a dict to print per sampling configuration config
            eogunit_config_dict = {}
            for config in eogunit_per_sub.unique():
                sub = eogunit_per_sub[eogunit_per_sub == config].index.tolist()
                eogunit_config_dict[config] = sub

            if len(eogunit_config_dict)>=1:
                # widget to select the configuration of interest
                config_eogunit_slider = mk_config_slider(value = 1, min = 1, max = len(eogunit_config_dict))
                
                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=eogunit_config_dict, param="Units"), i=config_eogunit_slider);
            else:
                print("No EOG unit found")
            #____________________________________________________________________________________________
            
            #%% 3.5 Inspect EOG signal inversion_________________________________________________________
            section35 = widgets.HTML(UI.subsection_html('3.5', 'Inspect EOG signal inversion'))
            display(section35)

            # select rows where the physical min is greater than the physical max
            df_eoginv = df_eog[df_eog['physical_min'] > df_eog['physical_max']]
            
            if not df_eoginv.empty:
                print('\n>>> Inverted polarity detected in EOGs! <<<')
                print(f'{df_eoginv.shape[0]} EOGs have an inverted polarity (from {df_eog.shape[0]} EOGs in {len(edf_files)} recordings)')
                print(df_eoginv[['subject', 'channel', 'dimension', 'physical_min', 'physical_max']])
            elif not not_applicable(df_eog, has_physical_bounds(df_eog), 'Inverted polarity', NO_BOUNDS):
                print('\n>>> No inverted polarity was detected in EOGs <<<')
            _merge_save(df_eoginv, f'{summary_path}/EOG_inverted_polarity_edf.tsv')
            print(f'\nSaving informations from inverted polarity EOGs to:\n{summary_path}/EOG_inverted_polarity_edf.tsv \n(will be empty if no inverted polarity)')
            #____________________________________________________________________________________________

            #%% 3.6 Inspect EOG dynamic range and resolution_____________________________________________
            section36 = widgets.HTML(UI.subsection_html('3.6', 'Inspect EOG dynamic range and resolution'))
            display(section36)
            #---------------------------
            section361 = widgets.HTML(UI.markdown_html("""
            <h4>Dynamic range</h4>
            """))
            display(section361)

            dr_eogthres = widgets.BoundedFloatText(
                value=400,
                min=0,
                max=5000,
                step=0.1,
                style={'description_width': '200px'},  # augmente la largeur de la description
                layout=widgets.Layout(width='270px'),   # ajuste la taille totale du widget si besoin
                description='Dynamic range threshold (µV):',
                disabled=False
            );

            def check_bad_eogdr(threshold):
                dr_mask = (df_eog['res_theoretical']*pow(2,16) <= threshold) & has_physical_bounds(df_eog)
                bad_dr = df_eog[dr_mask]
                
                if not bad_dr.empty:
                    print(f'\n>>> Dynamic range <= {threshold} µV detected in EOGs! <<<\n')
                    print(f'{bad_dr.shape[0]} EOGs detected (from {df_eog.shape[0]} EOGs in {len(edf_files)} recordings)')
                    print(bad_dr[['subject', 'channel', 'dimension', 'physical_min', 'physical_max', 'res_theoretical']])
                elif not not_applicable(df_eog, has_physical_bounds(df_eog), 'Dynamic range', NO_BOUNDS):
                    print(f'\n>>> No EOG with a dynamic range <= {threshold} µV was detected! <<<')
                _merge_save(bad_dr, f'{summary_path}/EOG_bad_dynamic_range_edf.tsv')
                print(f'\nSaving informations from bad dynamic range EOGs to:\n{summary_path}/EOG_bad_dynamic_range_edf.tsv \n(will be empty if no bad resolution)')
            
            widgets.interact(check_bad_eogdr, threshold = dr_eogthres);
            #---------------------------
            #---------------------------
            section362 = widgets.HTML(UI.markdown_html("""
            <h4>Resolution</h4>
            """))
            display(section362)

            # res_theo have been converted to uV, but if dimension was not read or not indicated in the headers, it might not work. I might need to add something more robust
            eogr_thres = widgets.BoundedFloatText(
                value=0.1,
                min=0,
                max=10.0,
                step=0.1,
                style={'description_width': '150px'},  # augmente la largeur de la description
                layout=widgets.Layout(width='230px'),   # ajuste la taille totale du widget si besoin
                description='Resolution threshold (µV):',
                disabled=False
            );
            
            # define a function to interact with the widget
            def check_bad_eogres(threshold):
                r_mask = df_eog['res_theoretical'] >= threshold
                bad_res = df_eog[r_mask]
                
                if not bad_res.empty:
                    print(f'\n>>> EOGs with a resolution >= {threshold} µV detected! <<<')
                    print(f'{bad_res.shape[0]} EOGs detected (from {df_eog.shape[0]} EOGs in {len(edf_files)} recordings)')
                    print(bad_res[['subject', 'channel', 'dimension', 'physical_min', 'physical_max', 'res_theoretical']])
                elif not not_applicable(df_eog, df_eog['res_theoretical'].notna(), 'Resolution', FLOAT_SAMPLES):
                    print(f'\n>>> No EOG with a resolution >= {threshold} µV was detected! <<<')
                _merge_save(bad_res, f'{summary_path}/EOG_bad_resolution_edf.tsv')
                print(f'\nSaving informations from bad resolution EOGs to:\n{summary_path}/EOG_bad_resolution_edf.tsv \n(will be empty if no bad resolution)')
            
            widgets.interact(check_bad_eogres, threshold=eogr_thres);
            #---------------------------
            #____________________________________________________________________________________________

        if include_ecg:
            #%% 4. Select only the ECGs__________________________________________________________________
            #____________________________________________________________________________________________
            section4 = widgets.HTML(UI.section_html(4, 'Inspect ECG', """
            <p>This section follows the same structure, logic, and outputs as section 2. Inspect EEG.
            <br>Hence, the code cells are not commented (please refer to section 2. if you need a refresher).
            </p>
            """))
            display(section4)

            # select only ECGs and return a warning if the number of participant is smaller/higher
            mask_ecg = df_full['transducer_type'].str.contains(r'\bECG|\bEKG', case = False, na=False) | df_full['channel'].str.contains(r'ecg|ekg', case = False, na=False) # create a mask that returns true for lines containing either ecg in the channel column
            df_ecg = df_full[mask_ecg]
            
            # Check if the number of participants with only ECG is the same as df_full. 
            # If not, it might be because the transducer type was no correctly detected. 
            # One possibility is to add the type of transducer to the condition line 2 of this cell.
            if len(df_full['subject'].unique()) > len(df_ecg['subject'].unique()):
                # identify missing subjects
                missing_sub = set(df_full['subject'].unique()) - set(df_ecg['subject'].unique())
                print('\n!!! There is less participants in the dataset with only ECGs !!!')
                print(f'Missing participants: {missing_sub}')
                print("\nEither these participants don't have ECGs.")
                print("Or the transducer type was not correctly detected.")
                # get df of missing sub to save and inspect
                df_ecgmiss = df_full[df_full['subject'].isin(missing_sub)]
                _merge_save(df_ecgmiss, f'{summary_path}/ECG_missing_edf.tsv')
                print(f'\nSaving informations from missing participants to:\n{summary_path}/ECG_missing_edf.tsv')
                print('Please inspect the file, and specifically the column transducer_type')
            elif len(df_full['subject'].unique()) < len(df_ecg['subject'].unique()):
                print('\n!!! There is more participants in the dataset with only ECGs !!!')
                print('This should not be the case.')
                print('Please inspect what is happening in a code editor (spyder..), or ask Yvan.')
                more_sub = set(df_ecg['subject'].unique()) - set(df_full['subject'].unique())
                df_more = df_ecg[df_ecg['subject'].isin(more_sub)]
                _merge_save(df_more, f'{summary_path}/ECG_suspect_edf.tsv')
                print(f'\nSaving informations from suspect participants to:\n{summary_path}/ECG_suspect_edf.tsv')
            
            # saving info from ECG
            _merge_save(df_ecg, f'{summary_path}/ECG_summary_table.tsv')
            print(f'\nSaving informations from ECGs to:\n{summary_path}/ECG_summary_table.tsv')
            #___________________________________________________________________________________________
            
            #%% 4.1 Inspect ECG configurations___________________________________________________________
            section41 = widgets.HTML(UI.subsection_html('4.1', 'Inspect ECG configurations'))
            display(section41)

            # get the ECGs configuration per participant 
            ecg_per_sub = df_ecg.groupby('subject')['channel'].apply(lambda x: tuple(sorted(set(x))))
            
            # identify the ECG configuration of each participant and store them in a dict to print per ECG config
            ecg_config_dict = {}
            for config in ecg_per_sub.unique():
                sub = ecg_per_sub[ecg_per_sub == config].index.tolist()
                ecg_config_dict[config] = sub
            
            if len(ecg_config_dict) > 1:
                print('\n>>> There is multiple ECG configurations in your dataset! <<<')    
                print(f'\n\tNumber of different ECG configuration: {len(ecg_config_dict)}\n')
            else:
                print('\n>>> There is only one ECG configuration in your dataset! <<<')

            if len(ecg_config_dict)>=1:
                # widget to select the configuration of interest
                config_ecg_slider = mk_config_slider(value = 1, min = 1, max = len(ecg_config_dict))
                
                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=ecg_config_dict, param="Channels"), i=config_ecg_slider);
            else:
                print("No ECG configuration found")
            #___________________________________________________________________________________________
            
            #%% 4.2 Inspect ECG sampling frequencies____________________________________________________
            section42 = widgets.HTML(UI.subsection_html('4.2', 'Inspect ECG sampling frequencies'))
            display(section42)
            
            # the sampling frequency configuration
            ecgsf_per_sub = df_ecg.groupby('subject')['sampling_frequency'].apply(lambda x: tuple(sorted(set(x))))
            # identify the sampling frequency configuration of each participant and store them in a dict to print per sampling configuration config
            ecgsf_config_dict = {}
            for config in ecgsf_per_sub.unique():
                sub = ecgsf_per_sub[ecgsf_per_sub == config].index.tolist()
                ecgsf_config_dict[config] = sub
            
            # print info per sf configuration (maybe print it only for multiple config)
            if len(ecgsf_config_dict) > 1:
                print('\n>>> There is multiple sampling frequency for ECGs in your dataset! <<<')    
                print(f'\n\tNumber of different sampling frequency configuration: {len(ecgsf_config_dict)}\n')
                print('Quick overlook of the ECGs associated to sampling frequencies:')
                for s, sf in enumerate(df_ecg['sampling_frequency'].unique()):
                    # select only rows with the current sf
                    df_sf = df_ecg[df_ecg['sampling_frequency'] == sf].copy()
                    print(f'\n{sf} Hz: {df_sf["channel"].unique()}')
            else:
                print(f'\n>>> There is only one sampling frequency for ECGs in your dataset: {df_ecg["sampling_frequency"].unique()} <<<')

            if len(ecgsf_config_dict):
                # widget to select the configuration of interest
                config_ecgsf_slider = mk_config_slider(value = 1, min = 1, max = len(ecgsf_config_dict))
                
                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=ecgsf_config_dict, param="Sampling frequencies"), i=config_ecgsf_slider);
            else:
                print("No ECG sampling frequency")
            #___________________________________________________________________________________________
            
            #%% 4.3 Inspect ECG filters_________________________________________________________________
            section43 = widgets.HTML(UI.subsection_html('4.3', 'Inspect ECG filters'))
            display(section43)

            if len(df_ecg['highpass'].unique())+len(df_ecg['lowpass'].unique())+len(df_ecg['notch'].unique()) == 3:
                print('\n>>> All ECGs have the same filters! <<<')
            elif len(df_ecg['highpass'].unique())+len(df_ecg['lowpass'].unique())+len(df_ecg['notch'].unique()) > 3:
                print('\n>>> Filters are not fully consistent across the dataset! <<<')
            else:
                print('\n>>> There may have been a problem in reading the filters. Here is the output: <<<')
            
            # Get the list of participants with different filtering parameters
            # 1st replace NaN because groupby does not like NaN
            df_ecgfilt = df_ecg.copy()
            df_ecgfilt[['lowpass', 'highpass', 'notch']] = df_ecgfilt[['lowpass', 'highpass', 'notch']].fillna('missing')
            
            config_ecgfilters = (
                df_ecgfilt.groupby(['lowpass', 'highpass', 'notch'])['subject']
                .apply(lambda x: sorted(set(x)))
                .reset_index(name = 'subjects')
            )

            # print filter configuration
            print(f'\n\tNumber of different ECG filters configurations: {len(config_ecgfilters)}\n')

            if len(config_ecgfilters)>=1:
                # widget to select the configuration of interest
                config_ecgfilter_slider = mk_config_slider(value = 1, min = 1, max = len(config_ecgfilters))
                
                # function to rpint filters configurations
                def print_ecgfilters(config_slider):
                    # get the info from the dataframe
                    idx = config_slider - 1
                    sID = config_ecgfilters.iloc[idx]['subjects']
                    hpass = config_ecgfilters.iloc[idx]['highpass']
                    lpass = config_ecgfilters.iloc[idx]['lowpass']
                    notch = config_ecgfilters.iloc[idx]['notch']
                    
                    # print info
                    print(f'Selected configuration: # {config_slider}')
                    print(f'\tFilters configuration: highpass: {hpass}; lowpass: {lpass}; notch: {notch}')
                    print(f'\t{len(sID)} participants: {sID}')
                
                widgets.interact(print_ecgfilters, config_slider = config_ecgfilter_slider);
            else:
                print("No ECG filters")
            #___________________________________________________________________________________________
            
            #%% 4.4 Inspect ECG units___________________________________________________________________
            section44 = widgets.HTML(UI.subsection_html('4.4', 'Inspect ECG units'))
            display(section44)

            if len(df_ecg['dimension'].unique()) == 1:
                print(f'\n>>> All ECGs have the same unit: {df_ecg["dimension"].unique()} <<<\n')
            elif len(df_ecg['dimension'].unique()) > 1:
                print('\n>>> Multiple units were found for ECGs! <<<')
                print(f'\n\tNumber of different units configurations: {len(df_ecg["dimension"].unique())}\n')
                print('Quick overlook of ECGs associated to units:')
                for u, unit in enumerate(df_ecg['dimension'].unique()):
                    # select only rows with the current sf
                    df_unit = df_ecg[df_ecg['dimension'] == unit].copy()
                    print(f'\n{unit}: {df_unit["channel"].unique()}')
                
            
            # print the different configuration of units 
            # if info about sf configuration is needed
            ecgunit_per_sub = df_ecg.groupby('subject')['dimension'].apply(lambda x: tuple(sorted(set(x))))
            ecg_per_unit = df_ecg.groupby('dimension')['channel'].apply(lambda x: tuple(sorted(set(x))))
            # identify the sampling frequency configuration of each participant and store them in a dict to print per sampling configuration config
            ecgunit_config_dict = {}
            for config in ecgunit_per_sub.unique():
                sub = ecgunit_per_sub[ecgunit_per_sub == config].index.tolist()
                ecgunit_config_dict[config] = sub

            if len(ecgunit_config_dict)>=1:
                # widget to select the configuration of interest
                config_ecgunit_slider = mk_config_slider(value = 1, min = 1, max = len(ecgunit_config_dict))
                
                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=ecgunit_config_dict, param="Units"), i=config_ecgunit_slider);
            else:
                print("No ECG unit found")
            #___________________________________________________________________________________________
            
            #%% 4.5 Inspect ECG signal inversion________________________________________________________
            section45 = widgets.HTML(UI.subsection_html('4.5', 'Inspect ECG signal inversion'))
            display(section45)

            # select rows where the physical min is greater than the physical max
            df_ecginv = df_ecg[df_ecg['physical_min'] > df_ecg['physical_max']]
            
            if not df_ecginv.empty:
                print('\n>>> Inverted polarity detected in ECGs! <<<')
                print(f'{df_ecginv.shape[0]} ECGs have an inverted polarity (from {df_ecg.shape[0]} ECGs in {len(edf_files)} recordings)')
                print(df_ecginv[['subject', 'channel', 'dimension', 'physical_min', 'physical_max']])
            elif not not_applicable(df_ecg, has_physical_bounds(df_ecg), 'Inverted polarity', NO_BOUNDS):
                print('\n>>> No inverted polarity was detected in ECGs <<<')
            _merge_save(df_ecginv, f'{summary_path}/ECG_inverted_polarity_edf.tsv')
            print(f'\nSaving informations from inverted polarity ECGs to:\n{summary_path}/ECG_inverted_polarity_edf.tsv \n(will be empty if no inverted polarity)')
            #____________________________________________________________________________________________

            #%% 4.6 Inspect ECG dynamic range and resolution_____________________________________________
            section46 = widgets.HTML(UI.subsection_html('4.6', 'Inspect ECG dynamic range and resolution'))
            display(section46)
            #---------------------------
            section461 = widgets.HTML(UI.markdown_html("""
            <h4>Dynamic range</h4>
            """))
            display(section461)

            ecgdr_thres = widgets.BoundedFloatText(
                value=400,
                min=0,
                max=5000,
                step=0.1,
                style={'description_width': '200px'},  # augmente la largeur de la description
                layout=widgets.Layout(width='270px'),   # ajuste la taille totale du widget si besoin
                description='Dynamic range threshold (µV):',
                disabled=False
            );
            
            
            def check_bad_ecgdr(threshold):
                dr_mask = (df_ecg['res_theoretical']*pow(2,16) <= threshold) & has_physical_bounds(df_ecg)
                bad_dr = df_ecg[dr_mask]
                
                if not bad_dr.empty:
                    print(f'\n>>> Dynamic range <= {threshold} µV detected in ECGs! <<<\n')
                    print(f'{bad_dr.shape[0]} ECGs detected (from {df_ecg.shape[0]} ECGs in {len(edf_files)} recordings)')
                    print(bad_dr[['subject', 'channel', 'dimension', 'physical_min', 'physical_max', 'res_theoretical']])
                elif not not_applicable(df_ecg, has_physical_bounds(df_ecg), 'Dynamic range', NO_BOUNDS):
                    print(f'\n>>> No ECG with a dynamic range <= {threshold} µV was detected! <<<')
                _merge_save(bad_dr, f'{summary_path}/ECG_bad_dynamic_range_edf.tsv')
                print(f'\nSaving informations from bad dynamic range ECGs to:\n{summary_path}/ECG_bad_dynamic_range_edf.tsv \n(will be empty if no bad resolution)')
            
            widgets.interact(check_bad_ecgdr, threshold = ecgdr_thres);
            #---------------------------
            #---------------------------
            section462 = widgets.HTML(UI.markdown_html("""
            <h4>Resolution</h4>
            """))
            display(section462)

            # res_theo have been converted to uV, but if dimension was not read or not indicated in the headers, it might not work. I might need to add something more robust
            ecgr_thres = widgets.BoundedFloatText(
                value=0.1,
                min=0,
                max=10.0,
                step=0.1,
                style={'description_width': '150px'},  # augmente la largeur de la description
                layout=widgets.Layout(width='230px'),   # ajuste la taille totale du widget si besoin
                description='Resolution threshold (µV):',
                disabled=False
            );
            
            # define a function to interact with the widget
            def check_bad_ecgres(threshold):
                r_mask = df_ecg['res_theoretical'] >= threshold
                bad_res = df_ecg[r_mask]
                
                if not bad_res.empty:
                    print(f'\n>>> ECGs with a resolution >= {threshold} µV detected! <<<')
                    print(f'{bad_res.shape[0]} ECGs detected (from {df_ecg.shape[0]} ecgs in {len(edf_files)} recordings)')
                    print(bad_res[['subject', 'channel', 'dimension', 'physical_min', 'physical_max', 'res_theoretical']])
                elif not not_applicable(df_ecg, df_ecg['res_theoretical'].notna(), 'Resolution', FLOAT_SAMPLES):
                    print(f'\n>>> No ECG with a resolution >= {threshold} µV was detected! <<<')
                _merge_save(bad_res, f'{summary_path}/ECG_bad_resolution_edf.tsv')
                print(f'\nSaving informations from bad resolution ECGs to:\n{summary_path}/ECG_bad_resolution_edf.tsv \n(will be empty if no bad resolution)')
            
            widgets.interact(check_bad_ecgres, threshold=ecgr_thres);
            #____________________________________________________________________________________________

        if include_emg:
            #%% 5. Select only the EMGs__________________________________________________________________
            #____________________________________________________________________________________________
            section5 = widgets.HTML(UI.section_html(5, 'Inspect EMG', """
            <p>This section follows the same structure, logic, and outputs as section 2. Inspect EEG.
            <br>Hence, the code cells are not commented (please refer to section 2. if you need a refresher).
            </p>
            """))
            display(section5)

            # select only EMGs and return a warning if the number of participant is smaller/higher
            mask_emg = df_full['transducer_type'].str.contains(r'\bEMG', case = False, na=False) | df_full['channel'].str.contains(r'emg|chin|menton', case = False, na=False) # create a mask that returns true for lines containing either emg in the channel column
            df_emg = df_full[mask_emg]

            # Check if the number of participants with only EMG is the same as df_full. 
            # If not, it might be because the transducer type was no correctly detected. 
            # One possibility is to add the type of transducer to the condition line 2 of this cell.
            if len(df_full['subject'].unique()) > len(df_emg['subject'].unique()):
                # identify missing subjects
                missing_sub = set(df_full['subject'].unique()) - set(df_emg['subject'].unique())
                print('\n!!! There is less participants in the dataset with only EMGs !!!')
                print(f'Missing participants: {missing_sub}')
                print("\nEither these participants don't have EMGs.")
                print("Or the transducer type was not correctly detected.")
                # get df of missing sub to save and inspect
                df_emgmiss = df_full[df_full['subject'].isin(missing_sub)]
                _merge_save(df_emgmiss, f'{summary_path}/EMG_missing_edf.tsv')
                print(f'\nSaving informations from missing participants to:\n{summary_path}/EMG_missing_edf.tsv')
                print('Please inspect the file, and specifically the column transducer_type')
            elif len(df_full['subject'].unique()) < len(df_emg['subject'].unique()):
                print('\n!!! There is more participants in the dataset with only EMGs !!!')
                print('This should not be the case.')
                print('Please inspect what is happening in a code editor (spyder..), or ask Yvan.')
                more_sub = set(df_emg['subject'].unique()) - set(df_full['subject'].unique())
                df_more = df_emg[df_emg['subject'].isin(more_sub)]
                _merge_save(df_more, f'{summary_path}/EMG_suspect_edf.tsv')
                print(f'\nSaving informations from suspect participants to:\n{summary_path}/EMG_suspect_edf.tsv')

            # saving info from EMG
            _merge_save(df_emg, f'{summary_path}/EMG_summary_table.tsv')
            print(f'\nSaving informations from EMGs to:\n{summary_path}/EMG_summary_table.tsv')
            #___________________________________________________________________________________________

            #%% 5.1 Inspect EMG configurations___________________________________________________________
            section51 = widgets.HTML(UI.subsection_html('5.1', 'Inspect EMG configurations'))
            display(section51)

            # get the EMGs configuration per participant 
            emg_per_sub = df_emg.groupby('subject')['channel'].apply(lambda x: tuple(sorted(set(x))))

            # identify the EMG configuration of each participant and store them in a dict to print per EMG config
            emg_config_dict = {}
            for config in emg_per_sub.unique():
                sub = emg_per_sub[emg_per_sub == config].index.tolist()
                emg_config_dict[config] = sub

            if len(emg_config_dict) > 1:
                print('\n>>> There is multiple EMG configurations in your dataset! <<<')    
                print(f'\n\tNumber of different EMG configuration: {len(emg_config_dict)}\n')
            else:
                print('\n>>> There is only one EMG configuration in your dataset! <<<')

            if len(emg_config_dict)>=1:
                # widget to select the configuration of interest
                config_emg_slider = mk_config_slider(value = 1, min = 1, max = len(emg_config_dict))

                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=emg_config_dict, param="Channels"), i=config_emg_slider);
            else:
                print("No EMG configuration found")
            #___________________________________________________________________________________________

            #%% 5.2 Inspect EMG sampling frequencies____________________________________________________
            section52 = widgets.HTML(UI.subsection_html('5.2', 'Inspect EMG sampling frequencies'))
            display(section52)

            # the sampling frequency configuration
            emgsf_per_sub = df_emg.groupby('subject')['sampling_frequency'].apply(lambda x: tuple(sorted(set(x))))
            # identify the sampling frequency configuration of each participant and store them in a dict to print per sampling configuration config
            emgsf_config_dict = {}
            for config in emgsf_per_sub.unique():
                sub = emgsf_per_sub[emgsf_per_sub == config].index.tolist()
                emgsf_config_dict[config] = sub

            # print info per sf configuration (maybe print it only for multiple config)
            if len(emgsf_config_dict) > 1:
                print('\n>>> There is multiple sampling frequency for EMGs in your dataset! <<<')    
                print(f'\n\tNumber of different sampling frequency configuration: {len(emgsf_config_dict)}\n')
                print('Quick overlook of the EMGs associated to sampling frequencies:')
                for s, sf in enumerate(df_emg['sampling_frequency'].unique()):
                    # select only rows with the current sf
                    df_sf = df_emg[df_emg['sampling_frequency'] == sf].copy()
                    print(f'\n{sf} Hz: {df_sf["channel"].unique()}')
            else:
                print(f'\n>>> There is only one sampling frequency for EMGs in your dataset: {df_emg["sampling_frequency"].unique()} <<<')

            if len(emgsf_config_dict):
                # widget to select the configuration of interest
                config_emgsf_slider = mk_config_slider(value = 1, min = 1, max = len(emgsf_config_dict))

                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=emgsf_config_dict, param="Sampling frequencies"), i=config_emgsf_slider);
            else:
                print("No EMG sampling frequency")
            #___________________________________________________________________________________________

            #%% 5.3 Inspect EMG filters_________________________________________________________________
            section53 = widgets.HTML(UI.subsection_html('5.3', 'Inspect EMG filters'))
            display(section53)

            if len(df_emg['highpass'].unique())+len(df_emg['lowpass'].unique())+len(df_emg['notch'].unique()) == 3:
                print('\n>>> All EMGs have the same filters! <<<')
            elif len(df_emg['highpass'].unique())+len(df_emg['lowpass'].unique())+len(df_emg['notch'].unique()) > 3:
                print('\n>>> Filters are not fully consistent across the dataset! <<<')
            else:
                print('\n>>> There may have been a problem in reading the filters. Here is the output: <<<')

            # Get the list of participants with different filtering parameters
            # 1st replace NaN because groupby does not like NaN
            df_emgfilt = df_emg.copy()
            df_emgfilt[['lowpass', 'highpass', 'notch']] = df_emgfilt[['lowpass', 'highpass', 'notch']].fillna('missing')

            config_emgfilters = (
                df_emgfilt.groupby(['lowpass', 'highpass', 'notch'])['subject']
                .apply(lambda x: sorted(set(x)))
                .reset_index(name = 'subjects')
            )

            # print filter configuration
            print(f'\n\tNumber of different EMG filters configurations: {len(config_emgfilters)}\n')

            if len(config_emgfilters)>=1:
                # widget to select the configuration of interest
                config_emgfilter_slider = mk_config_slider(value = 1, min = 1, max = len(config_emgfilters))

                # function to rpint filters configurations
                def print_emgfilters(config_slider):
                    # get the info from the dataframe
                    idx = config_slider - 1
                    sID = config_emgfilters.iloc[idx]['subjects']
                    hpass = config_emgfilters.iloc[idx]['highpass']
                    lpass = config_emgfilters.iloc[idx]['lowpass']
                    notch = config_emgfilters.iloc[idx]['notch']

                    # print info
                    print(f'Selected configuration: # {config_slider}')
                    print(f'\tFilters configuration: highpass: {hpass}; lowpass: {lpass}; notch: {notch}')
                    print(f'\t{len(sID)} participants: {sID}')

                widgets.interact(print_emgfilters, config_slider = config_emgfilter_slider);
            else:
                print("No EMG filters")
            #___________________________________________________________________________________________

            #%% 5.4 Inspect EMG units___________________________________________________________________
            section54 = widgets.HTML(UI.subsection_html('5.4', 'Inspect EMG units'))
            display(section54)

            if len(df_emg['dimension'].unique()) == 1:
                print(f'\n>>> All EMGs have the same unit: {df_emg["dimension"].unique()} <<<\n')
            elif len(df_emg['dimension'].unique()) > 1:
                print('\n>>> Multiple units were found for EMGs! <<<')
                print(f'\n\tNumber of different units configurations: {len(df_emg["dimension"].unique())}\n')
                print('Quick overlook of EMGs associated to units:')
                for u, unit in enumerate(df_emg['dimension'].unique()):
                    # select only rows with the current sf
                    df_unit = df_emg[df_emg['dimension'] == unit].copy()
                    print(f'\n{unit}: {df_unit["channel"].unique()}')


            # print the different configuration of units 
            # if info about sf configuration is needed
            emgunit_per_sub = df_emg.groupby('subject')['dimension'].apply(lambda x: tuple(sorted(set(x))))
            emg_per_unit = df_emg.groupby('dimension')['channel'].apply(lambda x: tuple(sorted(set(x))))
            # identify the sampling frequency configuration of each participant and store them in a dict to print per sampling configuration config
            emgunit_config_dict = {}
            for config in emgunit_per_sub.unique():
                sub = emgunit_per_sub[emgunit_per_sub == config].index.tolist()
                emgunit_config_dict[config] = sub

            if len(emgunit_config_dict)>=1:
                # widget to select the configuration of interest
                config_emgunit_slider = mk_config_slider(value = 1, min = 1, max = len(emgunit_config_dict))

                # print the configuration selected
                # interact with the slider output through the printing function 
                widgets.interact(lambda i: print_config(i, config_dict=emgunit_config_dict, param="Units"), i=config_emgunit_slider);
            else:
                print("No EMG unit found")
            #___________________________________________________________________________________________

            #%% 5.5 Inspect EMG signal inversion________________________________________________________
            section55 = widgets.HTML(UI.subsection_html('5.5', 'Inspect EMG signal inversion'))
            display(section55)

            # select rows where the physical min is greater than the physical max
            df_emginv = df_emg[df_emg['physical_min'] > df_emg['physical_max']]

            if not df_emginv.empty:
                print('\n>>> Inverted polarity detected in EMGs! <<<')
                print(f'{df_emginv.shape[0]} EMGs have an inverted polarity (from {df_emg.shape[0]} EMGs in {len(edf_files)} recordings)')
                print(df_emginv[['subject', 'channel', 'dimension', 'physical_min', 'physical_max']])
            elif not not_applicable(df_emg, has_physical_bounds(df_emg), 'Inverted polarity', NO_BOUNDS):
                print('\n>>> No inverted polarity was detected in EMGs <<<')
            _merge_save(df_emginv, f'{summary_path}/EMG_inverted_polarity_edf.tsv')
            print(f'\nSaving informations from inverted polarity EMGs to:\n{summary_path}/EMG_inverted_polarity_edf.tsv \n(will be empty if no inverted polarity)')
            #____________________________________________________________________________________________

            #%% 5.6 Inspect EMG dynamic range and resolution_____________________________________________
            section56 = widgets.HTML(UI.subsection_html('5.6', 'Inspect EMG dynamic range and resolution'))
            display(section56)
            #---------------------------
            section561 = widgets.HTML(UI.markdown_html("""
            <h4>Dynamic range</h4>
            """))
            display(section561)

            emgdr_thres = widgets.BoundedFloatText(
                value=400,
                min=0,
                max=5000,
                step=0.1,
                style={'description_width': '200px'},  # augmente la largeur de la description
                layout=widgets.Layout(width='270px'),   # ajuste la taille totale du widget si besoin
                description='Dynamic range threshold (µV):',
                disabled=False
            );


            def check_bad_emgdr(threshold):
                dr_mask = (df_emg['res_theoretical']*pow(2,16) <= threshold) & has_physical_bounds(df_emg)
                bad_dr = df_emg[dr_mask]

                if not bad_dr.empty:
                    print(f'\n>>> Dynamic range <= {threshold} µV detected in EMGs! <<<\n')
                    print(f'{bad_dr.shape[0]} EMGs detected (from {df_emg.shape[0]} EMGs in {len(edf_files)} recordings)')
                    print(bad_dr[['subject', 'channel', 'dimension', 'physical_min', 'physical_max', 'res_theoretical']])
                elif not not_applicable(df_emg, has_physical_bounds(df_emg), 'Dynamic range', NO_BOUNDS):
                    print(f'\n>>> No EMG with a dynamic range <= {threshold} µV was detected! <<<')
                _merge_save(bad_dr, f'{summary_path}/EMG_bad_dynamic_range_edf.tsv')
                print(f'\nSaving informations from bad dynamic range EMGs to:\n{summary_path}/EMG_bad_dynamic_range_edf.tsv \n(will be empty if no bad resolution)')

            widgets.interact(check_bad_emgdr, threshold = emgdr_thres);
            #---------------------------
            #---------------------------
            section562 = widgets.HTML(UI.markdown_html("""
            <h4>Resolution</h4>
            """))
            display(section562)

            # res_theo have been converted to uV, but if dimension was not read or not indicated in the headers, it might not work. I might need to add something more robust
            emgr_thres = widgets.BoundedFloatText(
                value=0.1,
                min=0,
                max=10.0,
                step=0.1,
                style={'description_width': '150px'},  # augmente la largeur de la description
                layout=widgets.Layout(width='230px'),   # ajuste la taille totale du widget si besoin
                description='Resolution threshold (µV):',
                disabled=False
            );

            # define a function to interact with the widget
            def check_bad_emgres(threshold):
                r_mask = df_emg['res_theoretical'] >= threshold
                bad_res = df_emg[r_mask]

                if not bad_res.empty:
                    print(f'\n>>> EMGs with a resolution >= {threshold} µV detected! <<<')
                    print(f'{bad_res.shape[0]} EMGs detected (from {df_emg.shape[0]} emgs in {len(edf_files)} recordings)')
                    print(bad_res[['subject', 'channel', 'dimension', 'physical_min', 'physical_max', 'res_theoretical']])
                elif not not_applicable(df_emg, df_emg['res_theoretical'].notna(), 'Resolution', FLOAT_SAMPLES):
                    print(f'\n>>> No EMG with a resolution >= {threshold} µV was detected! <<<')
                _merge_save(bad_res, f'{summary_path}/EMG_bad_resolution_edf.tsv')
                print(f'\nSaving informations from bad resolution EMGs to:\n{summary_path}/EMG_bad_resolution_edf.tsv \n(will be empty if no bad resolution)')

            widgets.interact(check_bad_emgres, threshold=emgr_thres);
            #____________________________________________________________________________________________

        #%% end of the code section__________________________________________________________________
        #____________________________________________________________________________________________

# selecting a folder refreshes the info line; the scan runs only on the Run button
chooser.register_callback(_update_existing_info)
skip_existing.observe(lambda ch: _update_existing_info(), names='value')
run_button.on_click(run_inspection)

# Display in voila
display(chooser, types_box, skip_existing, existing_files_info, run_button, progress_row, out)

✅ Packages and functions successfully imported!


HTML(value='\n<hr style="height:4px; background-color:black; border:none;">\n<h2>1. Select your study folder</h…

FileChooser(path='C:\Users\yvan.nedelec\OneDrive - ICM\Documents\RE_yvan\projects\Dream-Toolkit\python\Inspect…

Output()

<hr style="height:4px; background-color:black; border:none;">

## References

- Kemp B, Värri A, Rosa AC, Nielsen KD, Gade J. A simple format for exchange of digitized polygraphic recordings.
  Electroencephalogr Clin Neurophysiol. 1992;82(5):391-3. doi:[10.1016/0013-4694(92)90009-7](https://doi.org/10.1016/0013-4694%2892%2990009-7)
- Kemp B, Olivan J. European data format 'plus' (EDF+), an EDF alike standard format for the exchange of
  physiological data. Clin Neurophysiol. 2003;114(9):1755-61. doi:[10.1016/S1388-2457(03)00123-8](https://doi.org/10.1016/S1388-2457%2803%2900123-8)
- The EDF and EDF+ specifications: https://www.edfplus.info/
- Troester MM, Quan SF, Berry RB, et al. The AASM manual for the scoring of sleep and associated events: rules,
  terminology and technical specifications. Version 3. Darien (IL): American Academy of Sleep Medicine; 2023
  (recommended PSG montage and filters).